# 🐉 Hydra on Google Colab
### Mixture of neural fields for *ab initio* heterogeneous cryo-EM reconstruction

<a href="https://colab.research.google.com/github/ts387/hydra/blob/main/notebooks/hydra_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Hydra](https://github.com/ml-struct-bio/hydra) (Levy\*, Raghu\*, Shustin\* *et al.*, NeurIPS 2024) reconstructs a cryo-EM dataset as a mixture of **K neural fields**. Each particle is assigned to one of *K* classes (**compositional** heterogeneity), and each class has its own latent space of conformations (**conformational** heterogeneity). Poses are estimated *ab initio*. Hydra is built on [DRGN-AI](https://github.com/ml-struct-bio/drgnai) and is driven by the same `drgnai` command-line tool.

This notebook installs Hydra into its own Python 3.10 environment, because Hydra needs Python < 3.11 and Colab ships a newer Python. It then takes you through a complete experiment:

| Step | What happens | Typical time (T4 GPU) |
|:--|:--|:--|
| **1. Setup** | GPU check, optional Google Drive, helper functions | < 1 min |
| **2. Install** | isolated Python 3.10 env (via `uv`), PyTorch, Hydra, self-test | 2–4 min |
| **3. Data** | **A:** synthetic 3-class demo with ground truth, **or B:** your own particles (.mrcs / .star / .cs / .txt) | 1–15 min |
| **4. Configure** | pick a preset, adjust the key parameters, write `configs.yaml` | instant |
| **5. Train** | `drgnai train` in the background with a live dashboard (+ TensorBoard) | 10 min – many hours |
| **6. Analyze** | `drgnai analyze` for every class: PCA (UMAP when `Z_DIM > 2`), k-means and trajectory volumes | minutes |
| **7. Explore** | class assignments, accuracy against ground truth, latent spaces, 3D volume viewer | interactive |
| **8. Filter & export** | keep particles from selected classes, then download or save to Drive | minutes |

### How to use
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (or better). Training needs a GPU; on a CPU runtime you can still prepare data and explore results trained earlier.
2. Run the cells from top to bottom (▶ button or `Shift+Enter`). Every cell has a small form: change the fields, then run the cell. Double-click a form to see its code.
3. **Quick start:** keep all defaults and use **Runtime ▸ Run all**. This trains Hydra on a small synthetic 3-class mixture (roughly half an hour of training on a T4 by our estimate; the training dashboard shows a measured estimate once it runs) and then shows how well it recovered the classes.

> 💡 Colab erases `/content` when the runtime disconnects. For real datasets, tick **`USE_GOOGLE_DRIVE`** in step 1 so the data, checkpoints and results persist. You can then **resume** an interrupted training run in step 5.

---
## 1️⃣ Setup

In [ ]:
#@title 1. Setup: GPU check, storage location and helper functions { display-mode: "form" }
#@markdown **Where should data and results go?** Mount Google Drive to keep them if the runtime disconnects (recommended for anything but the demo).
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
WORK_DIR = "hydra_colab"  #@param {type:"string"}
#@markdown <small>A relative `WORK_DIR` is created inside `MyDrive/` when Drive is mounted, otherwise inside `/content/`. The Python environment and code always live on the fast local disk (`/content`).</small>

import os, sys, re, json, glob, time, shutil, shlex, pickle, signal, subprocess, datetime, io, textwrap, hashlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, HTML, Image

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------------------------------------------------------- storage --
LOCAL = Path(os.environ.get("HYDRA_COLAB_LOCAL", "/content" if IN_COLAB else str(Path.cwd())))
if WORK_DIR.startswith("/content/drive") and IN_COLAB:
    USE_GOOGLE_DRIVE = True          # an explicit Drive path needs the Drive mounted
if USE_GOOGLE_DRIVE and IN_COLAB:
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    _base = Path("/content/drive/MyDrive")
else:
    if USE_GOOGLE_DRIVE:
        print("⚠️ Google Drive is only available inside Colab; using local storage.")
    _base = LOCAL
WORK = Path(WORK_DIR) if os.path.isabs(WORK_DIR) else _base / WORK_DIR
for _d in ("data", "experiments", "uploads", "exports"):
    (WORK / _d).mkdir(parents=True, exist_ok=True)

ENV_DIR = LOCAL / "hydra-env"           # isolated Python 3.10 environment
REPO_DIR = LOCAL / "hydra"              # Hydra source code
SCRATCH = LOCAL / "hydra-scratch"       # fast local copies of particle stacks
SCRATCH.mkdir(parents=True, exist_ok=True)
BIN = ENV_DIR / "bin"
PY, CRYODRGN = BIN / "python", BIN / "cryodrgn"
DRGNAI = BIN / "drgnai-colab"          # `drgnai` wrapper written in step 2 (also works on CPU-only runtimes)
STATE_FILE = WORK / "notebook_state.json"
q = shlex.quote

# ------------------------------------------------------- kernel packages --
try:
    import mrcfile  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mrcfile"], check=True)
    import mrcfile  # noqa: F401


# ----------------------------------------------------------------- state --
def load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {"datasets": {}, "experiments": {}}


def save_state(state):
    tmp = STATE_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2))
    tmp.replace(STATE_FILE)


def get_dataset(name=""):
    st = load_state()
    name = name or st.get("active_dataset", "")
    if not name or name not in st["datasets"]:
        raise RuntimeError(f"Dataset {name!r} not found. Prepare one in step 3 first "
                           f"(known: {sorted(st['datasets'])}).")
    return name, st["datasets"][name]


def fingerprint(*parts):
    """Short hash identifying a dataset's contents, so experiments notice when their data was re-prepared."""
    return hashlib.sha1(json.dumps(parts, default=str).encode()).hexdigest()[:12]


def file_hash(path):
    return hashlib.sha1(Path(path).read_bytes()).hexdigest()[:12] if path else None


def exp_dataset(exp):
    """The dataset record an experiment was configured with (a snapshot, so later re-preparations don't change it)."""
    return exp.get("dataset_record") or get_dataset(exp["dataset"])[1]


def newest(paths):
    paths = list(paths)
    return max(paths, key=lambda p: p.stat().st_mtime) if paths else None


def get_experiment(name=""):
    st = load_state()
    name = name or st.get("active_experiment", "")
    if not name or name not in st["experiments"]:
        raise RuntimeError(f"Experiment {name!r} not found. Configure one in step 4 first "
                           f"(known: {sorted(st['experiments'])}).")
    return name, st["experiments"][name]


# ------------------------------------------------------- running commands --
def hydra_env():
    """Environment for subprocesses that run inside the Hydra Python 3.10 env."""
    env = {k: v for k, v in os.environ.items()
           if k not in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "MPLBACKEND")}
    env.update(PATH=f"{BIN}:{os.environ.get('PATH', '')}", VIRTUAL_ENV=str(ENV_DIR),
               MPLBACKEND="Agg", PYTHONUNBUFFERED="1", PYTHONNOUSERSITE="1",
               TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1", PYTHONWARNINGS="ignore")
    mode = load_state().get("ld_mode", "inherit")   # chosen by the CUDA self-test in step 2
    if mode == "filtered":   # keep only the NVIDIA driver directories (libcuda), drop other CUDA toolkits
        keep = [d for d in env.get("LD_LIBRARY_PATH", "").split(":") if d and "nvidia" in d and "cuda" not in d]
        env["LD_LIBRARY_PATH"] = ":".join(keep)
    elif mode == "none":
        env.pop("LD_LIBRARY_PATH", None)
    return env


def run(cmd, cwd=None, check=True, quiet=False, env=None):
    """Run a shell command (inside the Hydra env), streaming its output live."""
    if not quiet:
        print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env or hydra_env(),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:
        os.killpg(proc.pid, signal.SIGTERM)
        try:
            proc.wait(15)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        raise
    if check and proc.returncode != 0:
        if quiet:
            print("".join(lines[-60:]))
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {cmd}")
    return "".join(lines)


def hydra_python(code, quiet=True):
    """Run a snippet of Python code with the Hydra env's interpreter and return its stdout."""
    return run(f"{q(str(PY))} -c {q(textwrap.dedent(code))}", quiet=quiet)


def on_drive(path):
    return str(path).startswith("/content/drive")


def cache_locally(path, name):
    """Keep a fast local copy of a particle stack that lives on Google Drive. Returns (local, persistent)."""
    path = Path(path)
    if not on_drive(path):
        return path, None
    local = SCRATCH / name / path.name
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.exists() or local.stat().st_size != path.stat().st_size:
        print(f"Copying {path.name} ({path.stat().st_size / 1e9:.2f} GB) from Google Drive to local disk...")
        shutil.copy(path, str(local) + ".part")
        Path(str(local) + ".part").replace(local)
    return local, path


def ensure_particles(ds_name):
    """After a runtime reset the local copy is gone: restore it from the persistent copy on Drive."""
    ds = load_state()["datasets"][ds_name]
    local, source = Path(ds["particles"]), ds.get("particles_source")
    if source and not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        print(f"Restoring the local copy of {Path(source).name} from Google Drive...")
        shutil.copy(source, str(local) + ".part")
        Path(str(local) + ".part").replace(local)
    return ds


def require_install():
    if not DRGNAI.exists():
        raise RuntimeError("Hydra is not installed in this runtime yet. Run step 2 first.")


# Converts any particle source cryoDRGN can read (.mrcs/.star/.cs/.txt, any MRC mode) into a single
# float32 .mrcs, optionally Fourier-cropped to a smaller box. Runs inside the Hydra env.
CONVERT_SCRIPT = """
import sys, numpy as np, torch
from cryodrgn.source import ImageSource
from cryodrgn.mrcfile import MRCHeader
from cryodrgn import fft
src_path, out, box, datadir, apix_out = sys.argv[1], sys.argv[2], int(sys.argv[3]), (sys.argv[4] or None), float(sys.argv[5])
src = ImageSource.from_file(src_path, lazy=True, datadir=datadir)
D = src.D
box = min(box or D, D)
assert box % 2 == 0, f"box size {box} must be even"
s, e = (D - box) // 2, (D + box) // 2
hdr = MRCHeader.make_default_header(nz=src.n, ny=box, nx=box, Apix=round(apix_out, 6), dtype=np.float32, is_vol=False)
with open(out, "wb") as f:
    hdr.write(f)
    for i, (_, chunk) in enumerate(src.chunks(1000)):
        x = torch.as_tensor(chunk).float()
        if box != D:
            x = fft.iht2_center(fft.ht2_center(x)[:, s:e, s:e])
        f.write(np.asarray(torch.as_tensor(x).cpu(), dtype=np.float32).tobytes())
        if i % 20 == 19:
            print(f"  {min((i + 1) * 1000, src.n)}/{src.n} images", flush=True)
print(f"Converted {src.n} images: {D} px -> {box} px (float32)")
"""


def convert_particles(src, out, box=0, datadir=None, apix_out=1.0):
    """Write `src` as one float32 .mrcs at `box` px (0 = keep) with voxel size `apix_out`, via a temporary file."""
    script = SCRATCH / "convert_particles.py"
    script.write_text(CONVERT_SCRIPT)
    tmp = Path(str(out) + ".part")
    run(f"{q(str(PY))} {q(str(script))} {q(str(src))} {q(str(tmp))} {int(box)} {q(datadir or '')} {apix_out}")
    tmp.replace(out)


# ------------------------------------------------------------- file I/O --
def load_pkl(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pkl(obj, path):
    """Pickle data for Hydra (numpy 1.26 in the Hydra env reads numpy 2 pickles fine)."""
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)


def read_mrc(path):
    """Return (array, voxel size in Å) of an .mrc/.mrcs file (Hydra writes non-standard headers)."""
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with mrcfile.open(path, permissive=True) as m:
            return np.asarray(m.data, dtype=np.float32).copy(), float(m.voxel_size.x)


def mrc_header(path):
    with mrcfile.mmap(path, permissive=True) as m:
        return tuple(int(v) for v in m.data.shape), float(m.voxel_size.x)


def write_mrc(path, data, apix=1.0):
    with mrcfile.new(path, overwrite=True) as m:
        m.set_data(np.asarray(data, dtype=np.float32))
        m.voxel_size = apix


# ----------------------------------------------------------- plotting --
def montage(images, ncols=8, titles=None, size=1.5, cmap="gray", suptitle=None):
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i], cmap=cmap)
            if titles is not None:
                ax.set_title(str(titles[i]), fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


def projections(vol, crop=0.8):
    """Background-subtracted projections of a volume along z, y and x (central `crop` fraction of the box)."""
    D = vol.shape[0]
    lo, hi = int(D * (1 - crop) / 2), D - int(D * (1 - crop) / 2)
    out = []
    for a in (0, 1, 2):
        p = vol.sum(axis=a)[lo:hi, lo:hi]
        out.append(np.clip(p - np.median(p), 0, None))
    return out


def fig_to_png(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=90, bbox_inches="tight")
    plt.close(fig)
    return buf.getvalue()


def fmt_dt(seconds):
    return str(datetime.timedelta(seconds=int(max(0, seconds))))


def fmt_rough(seconds):
    if seconds < 3600:
        return f"{max(1, round(seconds / 60))} min"
    return f"{seconds / 3600:.1f} h" if seconds < 48 * 3600 else f"{seconds / 86400:.1f} days"


# --------------------------------------------------- training helpers --
_TRAIN_PROCS = globals().get("_TRAIN_PROCS", {})
_EPOCH_RE = re.compile(r"Epoch: (-?\d+) finished in (?:(\d+) days?, )?(\d+):(\d+):([\d.]+); total loss = ([-+\w.]+)")
_PROGRESS_RE = re.compile(r"\[Train Epoch: (-?\d+)/(\d+)\] \[(\d+)/(\d+) particles\]")


def _training_pid_alive(pid, exp_dir):
    """True if `pid` is a live `drgnai train` process for this experiment (pids are reused after a VM reset)."""
    try:
        cmdline = Path(f"/proc/{int(pid)}/cmdline").read_bytes().replace(b"\0", b" ").decode(errors="ignore")
        state = Path(f"/proc/{int(pid)}/status").read_text().split("State:")[1].split()[0]
    except (FileNotFoundError, IndexError, TypeError, ValueError, ProcessLookupError):
        return False
    return state != "Z" and "drgnai" in cmdline and " train " in cmdline and str(exp_dir) in cmdline


def training_running(exp):
    proc = _TRAIN_PROCS.get(exp["dir"])
    if proc is not None:
        return proc.poll() is None
    return _training_pid_alive(exp.get("pid"), exp["dir"])


def output_dirs(exp_dir):
    """Current out/ plus older runs moved aside by this notebook (out.<timestamp>), oldest first."""
    exp_dir = Path(exp_dir)
    return sorted(exp_dir.glob("out.*")) + [exp_dir / "out"]


def latest_epoch(outdir):
    """Newest epoch with a saved checkpoint (weights.<E>.pkl) in one output folder."""
    eps = [int(m.group(1)) for f in Path(outdir).glob("weights.*.pkl") if (m := re.match(r"weights\.(\d+)\.pkl$", f.name))]
    return max(eps) if eps else None


def epoch_dir(exp, epoch):
    """Folder holding an epoch's outputs: out/, or after a resume possibly an older out.<time> folder."""
    for d in reversed(output_dirs(exp["dir"])):
        if (d / f"weights.{epoch}.pkl").exists():
            return d
    return Path(exp["dir"]) / "out"


def lineage_files(exp_dir, prefix):
    """{epoch: path} of <prefix>.<epoch>.pkl over out/ and the out.<time> folders of resumed runs (newest wins)."""
    files = {}
    for d in output_dirs(exp_dir):
        for f in d.glob(f"{prefix}.*.pkl"):
            m = re.fullmatch(rf"{re.escape(prefix)}\.(-?\d+)\.pkl", f.name)
            if m:
                files[int(m.group(1))] = f
    return files


def pick_epoch(exp, requested=-1):
    """Epoch to show/analyse: the requested one, else the analysed one (if still present), else the newest."""
    if requested >= 0:
        return requested
    a = exp.get("analyzed_epoch")
    if a is not None and (epoch_dir(exp, a) / f"weights.{a}.pkl").exists():
        return a
    best = latest_checkpoint(exp["dir"])
    return best[1] if best else None


def latest_checkpoint(exp_dir):
    best = None
    for d in output_dirs(exp_dir):
        e = latest_epoch(d) if d.exists() else None
        if e is not None and (best is None or e >= best[1]):
            best = (d / f"weights.{e}.pkl", e)
    return best


def parse_training_log(path, info=None):
    info = info or dict(loss={}, duration={}, phase="starting", cur_epoch=None, seen=None, n=None,
                        finished=False, tail=[])
    if not Path(path).exists():
        return info
    lines = Path(path).read_text(errors="ignore").splitlines()
    for ln in lines:
        if "Will pretrain" in ln: info["phase"] = "pretraining"
        elif "Will use pose search" in ln: info["phase"] = "hierarchical pose search"
        elif "Will use latent optimization" in ln: info["phase"] = "SGD refinement"
        elif "Training Starts Now" in ln: info["finished"] = False
        m = _EPOCH_RE.search(ln)
        if m:
            e = int(m.group(1))
            info["duration"][e] = int(m.group(2) or 0) * 86400 + int(m.group(3)) * 3600 + int(m.group(4)) * 60 + float(m.group(5))
            try:
                info["loss"][e] = float(m.group(6))
            except ValueError:
                info["loss"][e] = float("nan")
        m = _PROGRESS_RE.search(ln)
        if m:
            info["cur_epoch"], info["seen"], info["n"] = int(m.group(1)), int(m.group(3)), int(m.group(4))
        if ln.startswith("Finished in"):
            info["finished"] = True
    info["tail"] = lines[-8:]
    return info


def training_info(exp_dir):
    """Merge training.log of the current run with those of earlier (resumed-from) runs."""
    info = None
    for d in output_dirs(exp_dir):
        info = parse_training_log(d / "training.log", info)
    return info


_STAMP = r"\((\d{2}-\w{3}-\d{2} \d{2}:\d{2}:\d{2})\)"


def epoch_progress(exp_dir):
    """(seen, n, seconds left in the current epoch) from the timestamped console log, or None."""
    console = Path(exp_dir) / "train_console.log"
    if not console.exists():
        return None
    t_start, last = None, None
    for ln in console.read_text(errors="ignore").splitlines()[-400:]:
        m = re.search(_STAMP, ln)
        if not m:
            continue
        t = datetime.datetime.strptime(m.group(1), "%d-%b-%y %H:%M:%S")
        if "Will use" in ln or "Will pretrain" in ln:
            t_start, last = t, None
        p = _PROGRESS_RE.search(ln)
        if p and t_start is not None:
            last = (int(p.group(3)), int(p.group(4)), t)
    if not last or not t_start or last[0] == 0:
        return None
    seen, n, t = last
    rate = seen / max((t - t_start).total_seconds(), 1)
    return seen, n, max(n - seen, 0) / rate


def monitor(name="", refresh=20):
    name, exp = get_experiment(name)
    exp_dir = Path(exp["dir"])
    console, exit_file = exp_dir / "train_console.log", exp_dir / "train_exit_code"
    status = display(Markdown("⏳ waiting for training to start..."), display_id=True)
    plot = display(HTML(""), display_id=True)
    t_launch = exp.get("started", time.time())
    n_ps, total = exp.get("pose_search_epochs", 0), exp.get("total_epochs", 1)
    n_plotted = 0
    try:
        while True:
            alive = training_running(exp)
            info = training_info(exp_dir)
            done = sorted(e for e in info["loss"] if e >= 0)
            dur_ps = [info["duration"][e] for e in done if e < n_ps]
            dur_sgd = [info["duration"][e] for e in done if e >= n_ps]
            left_ps = max(0, n_ps - len(dur_ps)); left_sgd = max(0, total - n_ps - len(dur_sgd))
            if dur_sgd or not left_sgd:
                eta = fmt_dt(left_ps * (np.mean(dur_ps) if dur_ps else 0) + left_sgd * (np.mean(dur_sgd) if dur_sgd else 0))
            elif dur_ps:
                eta = f"{fmt_dt(left_ps * np.mean(dur_ps))} for pose search, then {left_sgd} faster SGD epochs"
            else:
                eta = "estimating..."
            cur = (done[-1] + 1) if done else (-1 if info["phase"] == "pretraining" else 0)
            if done and info["phase"] == "pretraining":
                info["phase"] = "training with fixed poses"
            ep = epoch_progress(exp_dir) if alive else None
            prog = f" · {ep[0]}/{ep[1]} particles, ≈ {fmt_dt(ep[2])} left in this epoch" if ep else ""
            g = gpu_info()
            gtxt = f"GPU {g['util']}% busy, {g['used'] / 1024:.1f}/{g['total'] / 1024:.0f} GB" if g else "running on CPU"
            last = f"{info['loss'][done[-1]]:.5f}" if done else "–"
            state_txt = "🟢 running" if alive else ("✅ finished" if info["finished"] else "⏹ stopped")
            log_txt = (exp_dir / "out" / "training.log").read_text(errors="ignore") if (exp_dir / "out" / "training.log").exists() else ""
            warn = ("🛑 **Hydra is training on the CPU although a GPU is attached** - stop it (5-c) and re-run step 2 "
                    "with `REINSTALL` ticked.  \n" if g and "Use cuda False" in log_txt else "")
            status.update(Markdown(
                warn + f"**{name}** — {state_txt} · phase: **{info['phase']}**  \n"
                + (f"epoch **{cur}**{prog} · " if alive else "")
                + f"completed {len(done)}/{total} epochs · last epoch loss {last}  \n"
                f"elapsed {fmt_dt(time.time() - t_launch)}" + (f" · remaining ≈ {eta} · {gtxt}" if alive else "") + "\n\n"
                "```\n" + "\n".join(info["tail"][-6:]) + "\n```"))
            if len(done) >= 2 and len(done) != n_plotted:
                n_plotted = len(done)
                fig, ax = plt.subplots(figsize=(6, 2.6))
                ax.plot(done, [info["loss"][e] for e in done], "o-", ms=3)
                if n_ps:
                    ax.axvspan(-0.5, n_ps - 0.5, color="orange", alpha=.15, label="pose search")
                    ax.legend(fontsize=8)
                ax.set_xlabel("epoch"); ax.set_ylabel("total loss"); ax.set_title("Training loss"); ax.grid(alpha=.3)
                plot.update(Image(fig_to_png(fig)))
            if not alive:
                break
            time.sleep(refresh)
    except KeyboardInterrupt:
        print("Stopped watching; training continues in the background. Re-attach with 5-b, stop it with 5-c.\n"
              "⚠️ Colab may shut down a runtime that has no running cell, which also ends training - keep 5-b "
              "running during long runs.")
        raise   # also stops any remaining "Run all" cells, which would otherwise use unfinished results
    code_ = exit_file.read_text().strip() if exit_file.exists() else "?"
    if code_ == "0":
        print(f"✅ Training finished. Newest checkpoint: epoch {latest_epoch(exp_dir / 'out')}. Continue with step 6.")
    else:
        print(f"❌ Training stopped (exit code {code_}). Last lines of {console}:\n")
        print("\n".join(console.read_text(errors="ignore").splitlines()[-40:]) if console.exists() else "(no console log)")
        hint = (" Exit code 137 means it was killed, usually for running out of memory (see Troubleshooting)."
                if code_ == "137" else "")
        raise RuntimeError(f"Training of '{name}' did not finish.{hint} Fix the problem, then re-run step 5 "
                           "(START = auto resumes from the newest checkpoint).")


def oriented(vol):
    """Flip the sign of a map if its density is negative (Hydra's sign depends on the particle contrast)."""
    v = vol - vol.mean()
    return -vol if (v ** 3).mean() < 0 else vol


# -------------------------------------------------------------- GPU info --
def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=10)
        if out.returncode == 0 and out.stdout.strip():
            name, tot, used, util = [s.strip() for s in out.stdout.strip().splitlines()[0].split(",")]
            return dict(name=name, total=int(tot), used=int(used), util=int(util))
    except (FileNotFoundError, subprocess.TimeoutExpired, ValueError):
        pass
    return None


_g = gpu_info()
if _g:
    print(f"✅ GPU: {_g['name']} ({_g['total'] / 1024:.0f} GB)")
else:
    display(Markdown("⚠️ **No GPU detected.** Use *Runtime ▸ Change runtime type ▸ T4 GPU*. Training needs a GPU "
                     "(even the demo would take about a day on a CPU); without one you can still prepare data and "
                     "analyse or explore results that were trained earlier."))
import psutil
print(f"🧠 RAM: {psutil.virtual_memory().total / 1e9:.1f} GB   "
      f"💾 free local disk: {shutil.disk_usage(str(LOCAL)).free / 1e9:.0f} GB")
print(f"📁 Work directory: {WORK}" + ("  (Google Drive)" if str(WORK).startswith("/content/drive") else ""))

---
## 2️⃣ Install Hydra

Hydra needs Python 3.9–3.10 and the 2024-era scientific stack it was written for (numpy < 2, cryoDRGN 3.4, PyTorch 2.5; 2.8 on Blackwell GPUs). The cell below builds that stack, pinned to versions that pass Hydra's own test suite, in a separate environment at `/content/hydra-env` using [`uv`](https://docs.astral.sh/uv/). Colab's own Python is left untouched, and the notebook calls Hydra's `drgnai` command-line tool from that environment.

The environment lives on the runtime's local disk. After a disconnect or a *Disconnect and delete runtime*, re-run steps 1 and 2. Your data and results in `WORK_DIR` are kept if you use Google Drive.

In [ ]:
#@title 2. Install Hydra into an isolated Python 3.10 environment (≈2–4 min) { display-mode: "form" }
HYDRA_REPO = "https://github.com/ts387/hydra"  #@param ["https://github.com/ts387/hydra", "https://github.com/ml-struct-bio/hydra"] {allow-input: true}
HYDRA_BRANCH = "main"  #@param {type:"string"}
TORCH_VERSION = "auto"  #@param ["auto", "2.5.1", "2.8.0"] {allow-input: true}
#@markdown <small>`auto` picks PyTorch 2.5.1 (CUDA 12.4 build; T4, L4, A100, H100) or 2.8.0 for Blackwell GPUs, which need CUDA ≥ 12.8. Newer PyTorch versions change `torch.load` defaults that Hydra relies on; the notebook sets `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1` to cover that.</small>
REINSTALL = False  #@param {type:"boolean"}

# A dependency set validated with Hydra's own test-suite (Hydra needs numpy < 2 and the 2024-era cryoDRGN 3.4).
PINNED = ["cryodrgn==3.4.3", "numpy==1.26.4", "pandas==1.5.3", "matplotlib==3.6.3", "seaborn==0.11.2",
          "pillow==9.5.0", "scipy==1.15.3", "scikit-learn==1.7.2", "healpy==1.19.0", "umap-learn==0.5.12",
          "numba==0.68.0", "llvmlite==0.50.0", "pynndescent==0.6.0", "tensorboard==2.21.0", "protobuf==7.36.2",
          "pyyaml==6.0.3", "nbformat==5.11.1"]

t_start = time.time()
if REINSTALL:
    shutil.rmtree(ENV_DIR, ignore_errors=True)
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    st = load_state(); st.pop("installed", None); save_state(st)

# Colab pins its own package versions for uv via these variables; they must not leak into our env.
uv_env = {k: v for k, v in os.environ.items()
          if k not in ("UV_CONSTRAINT", "UV_BUILD_CONSTRAINT", "UV_OVERRIDE", "UV_SYSTEM_PYTHON",
                       "UV_PYTHON", "VIRTUAL_ENV", "PYTHONPATH", "PYTHONHOME")}
uv_env["UV_LINK_MODE"] = "copy"

# 1) uv: a fast Python/package manager (preinstalled on some Colab images)
if shutil.which("uv") is None:
    run(f"{q(sys.executable)} -m pip install -q uv", env=dict(os.environ))

# 2) Python 3.10 environment: uv's standalone Python, or conda-forge's via micromamba as a fallback
if not PY.exists():
    try:
        run(f"uv venv --seed --python 3.10 --python-preference only-managed {q(str(ENV_DIR))}", env=uv_env)
    except RuntimeError:
        print("uv could not download Python 3.10; creating the environment with micromamba from conda-forge instead...")
        shutil.rmtree(ENV_DIR, ignore_errors=True)
        mm = LOCAL / "micromamba"
        mm.mkdir(exist_ok=True)
        run(f"curl -fsSL https://conda.anaconda.org/conda-forge/linux-64/micromamba-2.9.0-0.tar.bz2 "
            f"| tar -xj -C {q(str(mm))} bin/micromamba", env=dict(os.environ))
        run(f"{q(str(mm / 'bin' / 'micromamba'))} create -y -q -p {q(str(ENV_DIR))} -c conda-forge --override-channels "
            f"python=3.10 pip", env=dict(os.environ, MAMBA_ROOT_PREFIX=str(mm)))
else:
    print(f"✔ Python environment already exists at {ENV_DIR}")

# 3) Hydra source code (a git clone: the package version comes from git metadata)
if not (REPO_DIR / "pyproject.toml").exists():
    run(f"git clone --depth 50 --branch {q(HYDRA_BRANCH)} {q(HYDRA_REPO)} {q(str(REPO_DIR))}", env=dict(os.environ))
else:
    head = run(f"git -C {q(str(REPO_DIR))} rev-parse --short HEAD", quiet=True, env=dict(os.environ)).strip()
    print(f"✔ Hydra source already present at {REPO_DIR} ({head})")

# 4) PyTorch, pinned dependencies and Hydra itself
torch_version = TORCH_VERSION
if torch_version == "auto":
    try:
        cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.split()
    except FileNotFoundError:
        cap = []
    torch_version = "2.8.0" if cap and float(cap[0]) >= 10.0 else "2.5.1"
pip = f"uv pip install --python {q(str(PY))}"
run(f"{pip} torch=={torch_version}", env=uv_env)
run(f"{pip} {' '.join(q(p) for p in PINNED)} {q(str(REPO_DIR))}", env=uv_env)

# 5) `drgnai-colab`: runs Hydra's CLI; on a CPU-only runtime it maps GPU-saved checkpoints to the CPU
DRGNAI.write_text(f"""#!{PY}
import functools, sys, torch
if not torch.cuda.is_available():
    torch.load = functools.partial(torch.load, map_location="cpu")
from cryodrgnai.command_line import run_cryodrgn_ai
sys.argv[0] = "drgnai"
sys.exit(run_cryodrgn_ai())
""")
DRGNAI.chmod(0o755)

# 6) Self-tests: real CUDA work (cuBLAS, cuDNN, cuFFT) inside the Hydra env, then Hydra's own import test
print("\n— Self-test —")
CUDA_CHECK = """
import sys, torch, numpy, cryodrgn, cryodrgnai
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | numpy {numpy.__version__} | cryodrgn {cryodrgn.__version__}")
if torch.cuda.is_available():
    x = torch.randn(8, 3, 64, 64, device="cuda")
    y = torch.nn.functional.conv2d(x, torch.randn(4, 3, 3, 3, device="cuda"))
    z = torch.fft.fft2(x).abs().sum() + (x.flatten(1) @ x.flatten(1).T).sum() + y.sum()
    torch.cuda.synchronize(); z.item()
    print("CUDA OK:", torch.cuda.get_device_name(0), "| CUDA runtime", torch.version.cuda)
else:
    print("CUDA NOT AVAILABLE")
"""
st = load_state()
cuda_ok = None
if gpu_info():
    # Colab's LD_LIBRARY_PATH can point at its own (newer) CUDA libraries; fall back to filtering it if needed.
    for mode in ("inherit", "filtered", "none"):
        st["ld_mode"] = mode; save_state(st)
        try:
            out = hydra_python(CUDA_CHECK)
        except RuntimeError as e:
            out = str(e)
        if "CUDA OK" in out:
            break
        print(f"(CUDA check failed with LD_LIBRARY_PATH mode '{mode}', trying the next one)")
    cuda_ok = "CUDA OK" in out
    if not cuda_ok:
        st["ld_mode"] = "inherit"; save_state(st)
    print(out.strip())
else:
    print(hydra_python(CUDA_CHECK).strip())
out = run(f"{q(str(DRGNAI))} test", quiet=True)
print("drgnai test:", out.strip().splitlines()[-1] if out.strip() else "(no output)")
st["installed"] = {"repo": HYDRA_REPO, "branch": HYDRA_BRANCH, "torch": torch_version, "cuda_ok": cuda_ok}
save_state(st)
if cuda_ok is False:
    raise RuntimeError("Hydra is installed, but PyTorch cannot use the GPU from its environment "
                       f"(LD_LIBRARY_PATH={os.environ.get('LD_LIBRARY_PATH')}). Try REINSTALL, or TORCH_VERSION = 2.8.0.")
print(f"✅ Hydra is ready ({fmt_dt(time.time() - t_start)}).")

---
## 3️⃣ Data

Choose **one** of the two options below. Whichever cell you run *last* becomes the active dataset for the following steps (step 8-a can also make a particle subset active). You can prepare several datasets and switch between them in step 4.

* **3-A · Synthetic demo** generates a small dataset with known ground truth: several distinct "molecules", each with a hinged domain, random orientations, CTF and noise. It takes about 1 minute and is the fastest way to see Hydra working.
* **3-B · Your own particles** accepts a particle stack (`.mrcs`, RELION `.star`, cryoSPARC `.cs`, or a `.txt` list of stacks), CTF parameters, and optional poses. It can also download data from a URL and downsample it.

Hydra expects particles to be **already picked, extracted and centred**, with a box size that is **even**. Box sizes of 64–128 px are the practical range on Colab.

In [ ]:
#@title 3-A. Generate a synthetic mixture dataset (demo) { display-mode: "form" }
GENERATE_SYNTHETIC = True  #@param {type:"boolean"}
DATASET_NAME = "synthetic_mixture"  #@param {type:"string"}
N_PARTICLES = 2000  #@param {type:"integer"}
BOX_SIZE = 64  #@param [64, 96, 128] {type:"raw"}
PIXEL_SIZE = 4.0  #@param {type:"number"}
#@markdown Number of distinct molecules (= ground-truth classes) and whether each has a continuously moving domain:
N_TRUE_CLASSES = 3  #@param {type:"slider", min:1, max:4, step:1}
#@markdown <small>Set `N_CLASSES` in step 4 to the same number (or one more, to see what Hydra does with a spare class).</small>
CONFORMATIONAL_CHANGE = True  #@param {type:"boolean"}
#@markdown Signal-to-noise ratio (variance of signal / variance of noise; real data is ~0.01–0.1):
SNR = 0.1  #@param {type:"number"}
MAX_SHIFT_PIXELS = 0  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}

_fp = fingerprint("synthetic-v2", N_PARTICLES, BOX_SIZE, PIXEL_SIZE, N_TRUE_CLASSES, CONFORMATIONAL_CHANGE, SNR,
                  MAX_SHIFT_PIXELS, SEED)
_old = load_state()["datasets"].get(DATASET_NAME, {})
if not GENERATE_SYNTHETIC:
    print("Skipped (GENERATE_SYNTHETIC is off).")
elif _old.get("fingerprint") == _fp and Path(_old["particles"]).exists():
    st = load_state(); st["active_dataset"] = DATASET_NAME; save_state(st)
    print(f"Dataset '{DATASET_NAME}' already exists with these settings - reusing {Path(_old['particles']).parent}.")
else:
    _users = [n for n, e in load_state()["experiments"].items() if e.get("dataset") == DATASET_NAME]
    if _old and _users:
        display(Markdown(f"⚠️ Replacing dataset `{DATASET_NAME}`, which experiment(s) {', '.join(_users)} were trained on. "
                         "Re-run step 4 before training them again; their earlier results no longer match the files."))
    require_install()
    assert BOX_SIZE % 2 == 0, "BOX_SIZE must be even"
    rng = np.random.default_rng(SEED)
    D, apix = int(BOX_SIZE), float(PIXEL_SIZE)
    R = 0.30 * D * apix                     # molecule radius in Å
    sigma_atom = 1.2 * apix                 # pseudo-atom width in Å
    t0 = time.time()

    def _blob(center, radius, n):
        v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
        return np.asarray(center, float) * R + v * radius * R * rng.uniform(0, 1, (n, 1)) ** (1 / 3)

    def _axis_rot(axis, angle):
        a = np.asarray(axis, float) / np.linalg.norm(axis)
        K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
        return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K

    ring = lambda t: [0.62 * np.cos(t), 0.62 * np.sin(t), 0.0]
    MOLECULES = [  # (rigid blobs, flexible blobs, hinge point, hinge axis, max angle in degrees)
        ("dumbbell", [([0, 0, -0.55], .38, 260), ([0, 0, 0], .18, 60)], [([0, 0, .62], .32, 200)], [0, 0, .08], [1, 0, 0], 70),
        ("tripod", [([0, 0, 0], .28, 150), ([.68, 0, 0], .22, 110), ([-.34, .59, 0], .22, 110)],
                   [([-.34, -.59, 0], .24, 140)], [0, 0, 0], [0, 0, 1], 60),
        ("open ring", [(ring(t), .17, 55) for t in np.linspace(0, 1.35 * np.pi, 6)],
                      [(ring(t), .17, 55) for t in np.linspace(1.62 * np.pi, 1.82 * np.pi, 2)], ring(0), [0, 0, 1], 50),
        ("L-rod", [([x, 0, -.55], .17, 70) for x in np.linspace(-.6, .25, 4)] + [([.25, 0, -.2], .17, 70)],
                  [([.25, 0, .3], .2, 90), ([.25, 0, .62], .2, 90)], [.25, 0, -.2], [0, 1, 0], 75),
    ]
    mols = []
    for name, rigid, flex, hinge, axis, max_deg in MOLECULES[:N_TRUE_CLASSES]:
        rig = np.concatenate([_blob(c, r, n) for c, r, n in rigid])
        flx = np.concatenate([_blob(c, r, n) for c, r, n in flex])
        com = np.concatenate([rig, flx]).mean(0)
        mols.append(dict(name=name, rigid=rig - com, flex=flx - com, hinge=np.asarray(hinge) * R - com,
                         axis=axis, max_angle=np.deg2rad(max_deg) if CONFORMATIONAL_CHANGE else 0.0))

    def atoms(mol, conf):
        Rh = _axis_rot(mol["axis"], conf * mol["max_angle"])
        return np.concatenate([mol["rigid"], (mol["flex"] - mol["hinge"]) @ Rh.T + mol["hinge"]])

    def random_rotations(n):
        qt = rng.normal(size=(n, 4)); qt /= np.linalg.norm(qt, axis=1, keepdims=True)
        w, x, y, z = qt.T
        return np.stack([np.stack([1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)], -1),
                         np.stack([2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)], -1),
                         np.stack([2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)], -1)], 1)

    grid = (np.arange(D) - D // 2) * apix   # pixel centres in Å (origin at D/2, as in cryo-EM software)
    def gauss1d(c):                          # c: (n_atoms,) -> (n_atoms, D)
        return np.exp(-(grid[None, :] - c[:, None]) ** 2 / (2 * sigma_atom ** 2))

    def project(xyz):                        # analytic projection of Gaussian atoms along z
        return gauss1d(xyz[:, 1]).T @ gauss1d(xyz[:, 0])        # rows = y, columns = x

    def rasterize(xyz):                      # 3D density, axes (z, y, x)
        return np.einsum("az,ay,ax->zyx", gauss1d(xyz[:, 2]), gauss1d(xyz[:, 1]), gauss1d(xyz[:, 0]), optimize=True)

    def ctf2d(dfu, dfv, dfang, volt=300.0, cs=2.7, w=0.1, phase_shift=0.0):
        f = np.fft.fftshift(np.fft.fftfreq(D, d=apix))          # same frequency grid as Hydra (cycles/Å)
        fx, fy = np.meshgrid(f, f)
        volt, cs = volt * 1e3, cs * 1e7
        lam = 12.2639 / np.sqrt(volt + 0.97845e-6 * volt ** 2)
        ang, s2 = np.arctan2(fy, fx), fx ** 2 + fy ** 2
        df = 0.5 * (dfu + dfv + (dfu - dfv) * np.cos(2 * (ang - np.deg2rad(dfang))))
        gamma = 2 * np.pi * (-0.5 * df * lam * s2 + 0.25 * cs * lam ** 3 * s2 ** 2) - np.deg2rad(phase_shift)
        return np.sqrt(1 - w ** 2) * np.sin(gamma) - w * np.cos(gamma)

    N = int(N_PARTICLES)
    labels = rng.integers(0, N_TRUE_CLASSES, N)
    confs = rng.uniform(0, 1, N) if CONFORMATIONAL_CHANGE else np.zeros(N)
    rots = random_rotations(N)
    shifts = rng.uniform(-MAX_SHIFT_PIXELS, MAX_SHIFT_PIXELS, (N, 2)) if MAX_SHIFT_PIXELS > 0 else np.zeros((N, 2))
    dfu = rng.uniform(10000, 25000, N); dfv = dfu + rng.normal(0, 300, N); dfang = rng.uniform(0, 180, N)

    clean = np.empty((N, D, D), np.float32)
    for i in range(N):
        xyz = atoms(mols[labels[i]], confs[i]) @ rots[i].T
        xyz[:, :2] += shifts[i] * apix
        clean[i] = project(xyz)
    ctf_img = np.empty_like(clean)
    for s in range(0, N, 500):  # apply CTF in Fourier space
        sl = slice(s, min(s + 500, N))
        H = np.stack([ctf2d(a, b, c) for a, b, c in zip(dfu[sl], dfv[sl], dfang[sl])])
        F = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(clean[sl], axes=(-2, -1))), axes=(-2, -1))
        ctf_img[sl] = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(F * H, axes=(-2, -1))), axes=(-2, -1)).real
    noise_std = np.sqrt(ctf_img.var() / SNR)
    # store with inverted contrast ("white" particles), as RELION/cryoSPARC extract them by default
    particles = -(ctf_img + rng.normal(0, noise_std, ctf_img.shape)).astype(np.float32)

    ddir = WORK / "data" / DATASET_NAME
    ddir.mkdir(parents=True, exist_ok=True)
    write_mrc(ddir / "particles.mrcs", particles, apix)
    ctf = np.column_stack([np.full(N, D), np.full(N, apix), dfu, dfv, dfang,
                           np.full(N, 300.0), np.full(N, 2.7), np.full(N, 0.1), np.zeros(N)]).astype(np.float32)
    save_pkl(ctf, ddir / "ctf.pkl")
    save_pkl(labels.astype(np.int64), ddir / "labels.pkl")
    save_pkl(confs.astype(np.float32), ddir / "gt_conformation.pkl")
    # Hydra's pose convention: image = projection of the volume sampled at r @ R; a particle displaced by
    # +s pixels is stored as translation -s / (D + 1) (fraction of the Hartley lattice size)
    save_pkl((rots.astype(np.float32), (-shifts / (D + 1)).astype(np.float32)), ddir / "poses.pkl")
    gt_dir = ddir / "ground_truth_volumes"; gt_dir.mkdir(exist_ok=True)
    for k, m in enumerate(mols):
        for c in ([0.0, 0.5, 1.0] if CONFORMATIONAL_CHANGE else [0.0]):
            write_mrc(gt_dir / f"class{k}_{m['name'].replace(' ', '_')}_conf{c:.1f}.mrc", rasterize(atoms(m, c)), apix)

    st = load_state()
    st["datasets"][DATASET_NAME] = dict(
        particles=str(ddir / "particles.mrcs"), ctf=str(ddir / "ctf.pkl"), pose=str(ddir / "poses.pkl"),
        labels=str(ddir / "labels.pkl"), gt_conformation=str(ddir / "gt_conformation.pkl"),
        gt_volumes=str(gt_dir), datadir=None, ind=None, N=N, D=D, apix=apix,
        has_shifts=bool(MAX_SHIFT_PIXELS > 0), synthetic=True, source_file=str(ddir / "particles.mrcs"),
        class_names=[m["name"] for m in mols], fingerprint=_fp)
    st["active_dataset"] = DATASET_NAME
    save_state(st)

    print(f"✅ Wrote {N} particles ({D}×{D} px, {apix} Å/px) to {ddir} in {fmt_dt(time.time() - t0)}")
    print("   class sizes:", {m["name"]: int((labels == k).sum()) for k, m in enumerate(mols)})
    print(f"   → use N_CLASSES = {N_TRUE_CLASSES} in step 4")
    confs_shown = [0.0, 1.0] if CONFORMATIONAL_CHANGE else [0.0]
    # view each molecule along its thinnest direction (atom axes x,y,z = volume axes 2,1,0)
    view = [2 - int(np.argmin(atoms(m, 0.0).var(0))) for m in mols]
    montage([rasterize(atoms(m, c)).sum(axis=view[k]) for c in confs_shown for k, m in enumerate(mols)], ncols=len(mols),
            titles=[f"{m['name']}" + (" (conf 0)" if CONFORMATIONAL_CHANGE else "") for m in mols] +
                   ([f"{m['name']} (conf 1)" for m in mols] if CONFORMATIONAL_CHANGE else []),
            size=2.2, cmap="magma", suptitle="Ground-truth molecules (projections)")
    idx = rng.choice(N, 16, replace=False)
    montage(np.concatenate([clean[idx[:8]], particles[idx[:8]]]), ncols=8,
            titles=[f"clean · {mols[labels[i]]['name']}" for i in idx[:8]] + ["CTF + noise"] * 8,
            suptitle=f"Example particles (SNR {SNR})")

In [ ]:
#@title (optional) Upload files from your computer { display-mode: "form" }
#@markdown Uploads go to `WORK_DIR/uploads/`; refer to them in 3-B as `uploads/<file name>`. Browser uploads are slow; use Google Drive for anything above a few hundred MB. A `.star` file needs the `.mrcs` stacks it references as well.
UPLOAD_FILES = False  #@param {type:"boolean"}
if UPLOAD_FILES:
    if not IN_COLAB:
        print("Uploading only works inside Google Colab.")
    else:
        from google.colab import files
        for fname in files.upload(target_dir=str(WORK / "uploads")):
            path = WORK / "uploads" / Path(fname).name
            print(f"saved {path} ({path.stat().st_size / 1e6:.1f} MB) - use PARTICLES / CTF_PKL = 'uploads/{path.name}' in 3-B")
else:
    print("Skipped (tick UPLOAD_FILES to open the upload dialog).")

In [ ]:
#@title 3-B. Use your own particles { display-mode: "form" }
DATASET_NAME = "my_dataset"  #@param {type:"string"}
#@markdown **Particles**: a path or an `https://`/`ftp://` URL to download (`.tar`, `.tar.gz` and `.zip` archives are unpacked). Paths can be absolute (e.g. `/content/drive/MyDrive/project/Refine3D/job050/run_data.star`) or relative to `WORK_DIR` (e.g. `uploads/particles.mrcs`). Leave empty to skip this cell.
PARTICLES = ""  #@param {type:"string"}
#@markdown If an archive is downloaded, give the particle file's path *inside* the archive:
PARTICLES_IN_ARCHIVE = ""  #@param {type:"string"}
#@markdown Folder that the image paths inside a `.star`/`.cs` file are relative to: usually the RELION project folder (the one containing `Extract/`) or the cryoSPARC project folder (containing the `J…` job folders). Blank = the folder holding the `.star`/`.cs`.
DATADIR = ""  #@param {type:"string"}
#@markdown ---
#@markdown **CTF parameters**: parse them from the `.star`/`.cs` file, or point to an existing cryoDRGN-style `ctf.pkl`.
CTF_SOURCE = "parse from particle .star/.cs"  #@param ["parse from particle .star/.cs", "existing ctf .pkl"]
CTF_PKL = ""  #@param {type:"string"}
#@markdown <small>Only needed when the `.star`/`.cs` file lacks these fields (0 = read from file):</small>
ORIGINAL_BOX_SIZE = 0  #@param {type:"integer"}
ORIGINAL_PIXEL_SIZE = 0.0  #@param {type:"number"}
VOLTAGE_KV = 0.0  #@param {type:"number"}
SPHERICAL_ABERRATION_MM = 0.0  #@param {type:"number"}
AMPLITUDE_CONTRAST = 0.0  #@param {type:"number"}
#@markdown ---
#@markdown **Poses** (optional, only used with pose estimation `refine` or `fixed`):
POSE_SOURCE = "none"  #@param ["none", "parse from particle .star/.cs", "existing pose .pkl"]
POSE_PKL = ""  #@param {type:"string"}
CRYOSPARC_JOB_TYPE = "homogeneous refinement"  #@param ["homogeneous refinement", "heterogeneous refinement", "ab initio"]
#@markdown ---
#@markdown **Preprocessing**
DOWNSAMPLE_TO = 128  #@param [0, 64, 96, 128, 160, 192, 256] {type:"raw"}
#@markdown <small>Target box size in pixels (0, or a value at least your box size, = keep it). `.star`/`.cs`/`.txt` inputs and non-float32 stacks are converted into a single float32 `.mrcs` in `WORK_DIR/data/<name>/`. The CTF file stays valid: Hydra rescales the pixel size automatically.</small>
SUBSET_INDEX_PKL = ""  #@param {type:"string"}
#@markdown <small>Optional `.pkl` with an array of particle indices to keep (e.g. from step 8-a).</small>
MAX_PARTICLES = 0  #@param {type:"integer"}
#@markdown <small>Train on a random subset of at most this many particles (0 = all). Pose search time grows with the number of particles, so a subset of 20 000–50 000 is a good first look at a large dataset.</small>
COPY_TO_LOCAL_DISK = True  #@param {type:"boolean"}
#@markdown <small>Train from a copy of the stack on the runtime's local disk (much faster than reading from Google Drive; the copy is re-made automatically after a runtime reset).</small>


def _fetch(url, dest_dir):
    """Download (and unpack) into local scratch space; partial downloads are never mistaken for complete ones."""
    dest_dir.mkdir(parents=True, exist_ok=True)
    fname = dest_dir / url.split("?")[0].rstrip("/").split("/")[-1]
    if not fname.exists():
        part = Path(str(fname) + ".part")
        run(f"wget -c --progress=dot:giga -O {q(str(part))} {q(url)}", env=dict(os.environ))
        part.replace(fname)
    if re.search(r"\.(tar\.gz|tgz|tar|zip)$", fname.name):
        out = dest_dir / "extracted"
        if not out.exists():
            tmp = dest_dir / "extracted.part"
            shutil.rmtree(tmp, ignore_errors=True)
            tmp.mkdir()
            cmd = (f"unzip -q {q(str(fname))} -d {q(str(tmp))}" if fname.name.endswith(".zip")
                   else f"tar -xf {q(str(fname))} -C {q(str(tmp))}")
            run(cmd, env=dict(os.environ))
            tmp.replace(out)
        return out
    return fname


def _resolve(p):
    p = p.strip()
    if not p:
        return None
    if re.match(r"^(https?|ftp)://", p):
        return _fetch(p, SCRATCH / DATASET_NAME / "download")
    path = Path(p) if os.path.isabs(p) else WORK / p
    if not path.exists():
        raise FileNotFoundError(f"Not found: {path}" + ("" if os.path.isabs(p) else " (relative paths start in WORK_DIR)"))
    return path


if not PARTICLES.strip():
    print("Skipped (PARTICLES is empty).")
else:
    require_install()
    ddir = WORK / "data" / DATASET_NAME
    ddir.mkdir(parents=True, exist_ok=True)
    src = _resolve(PARTICLES)
    if src.is_dir():
        if not PARTICLES_IN_ARCHIVE:
            cands = sorted(str(p.relative_to(src)) for ext in ("*.star", "*.cs", "*.mrcs", "*.txt") for p in src.rglob(ext))
            raise ValueError("Downloaded an archive - set PARTICLES_IN_ARCHIVE to one of:\n  " + "\n  ".join(cands[:30]))
        src = src / PARTICLES_IN_ARCHIVE
    ext = src.suffix.lower()
    assert ext in (".mrcs", ".mrc", ".star", ".cs", ".txt"), f"Unsupported particle format: {ext}"
    datadir = str(_resolve(DATADIR)) if DATADIR.strip() else None
    print(f"Particles: {src}")

    # ---- CTF ----
    img_flags = (f" -D {ORIGINAL_BOX_SIZE}" if ORIGINAL_BOX_SIZE else "") + \
                (f" --Apix {ORIGINAL_PIXEL_SIZE}" if ORIGINAL_PIXEL_SIZE else "")
    if CTF_SOURCE.startswith("existing"):
        ctf_pkl = _resolve(CTF_PKL)
    else:
        ctf_pkl = ddir / "ctf.pkl"
        if ext == ".star":
            optics = "".join(f" {flag} {val}" for flag, val in (("--kv", VOLTAGE_KV), ("--cs", SPHERICAL_ABERRATION_MM),
                                                                 ("-w", AMPLITUDE_CONTRAST)) if val)
            run(f"{q(str(CRYODRGN))} parse_ctf_star {q(str(src))} -o {q(str(ctf_pkl))}{img_flags}{optics}", quiet=True)
        elif ext == ".cs":
            run(f"{q(str(CRYODRGN))} parse_ctf_csparc {q(str(src))} -o {q(str(ctf_pkl))}{img_flags}", quiet=True)
        else:
            raise ValueError("CTF parameters can only be parsed from a .star or .cs file; "
                             "set CTF_SOURCE to 'existing ctf .pkl' and give CTF_PKL.")
    ctf_arr = load_pkl(ctf_pkl)
    if np.isnan(ctf_arr[:, 8]).any():           # RELION 3.0 files often lack the phase-shift column
        print("No phase shift in the file - assuming 0°.")
        ctf_arr[:, 8] = np.nan_to_num(ctf_arr[:, 8]); save_pkl(ctf_arr, ctf_pkl)
    if np.isnan(ctf_arr).any():
        cols = ["box size", "pixel size", "defocusU", "defocusV", "defocus angle", "voltage", "Cs", "amplitude contrast"]
        missing = [c for i, c in enumerate(cols) if np.isnan(ctf_arr[:, i]).any()]
        raise ValueError(f"These CTF fields are missing from {src.name}: {missing}. Fill in the ORIGINAL_* / optics fields above.")
    D_orig = int(ctf_arr[0, 0])

    # ---- poses ----
    pose_pkl = None
    if POSE_SOURCE.startswith("existing"):
        pose_pkl = _resolve(POSE_PKL)
    elif POSE_SOURCE.startswith("parse"):
        pose_pkl = ddir / "poses.pkl"
        if ext == ".star":
            run(f"{q(str(CRYODRGN))} parse_pose_star {q(str(src))} -o {q(str(pose_pkl))}{img_flags}", quiet=True)
        elif ext == ".cs":
            job = {"heterogeneous refinement": " --hetrefine", "ab initio": " --abinit"}.get(CRYOSPARC_JOB_TYPE, "")
            run(f"{q(str(CRYODRGN))} parse_pose_csparc {q(str(src))} -D {D_orig} -o {q(str(pose_pkl))}{job}", quiet=True)
        else:
            raise ValueError("Poses can only be parsed from a .star or .cs file.")
    if pose_pkl is not None:
        pose = load_pkl(pose_pkl)
        if isinstance(pose, tuple) and np.abs(pose[1]).max() > 0.5:
            raise ValueError(f"The translations in {pose_pkl} look like pixels (max {np.abs(pose[1]).max():.1f}); Hydra expects "
                             "fractions of the box, as written by `cryodrgn parse_pose_*`. (Hydra's own pose.<epoch>.pkl "
                             "outputs are in pixels and cannot be fed back directly.)")

    # ---- particles: one float32 .mrcs at the target box size ----
    mode, D_stack = 2, D_orig
    if ext in (".mrcs", ".mrc"):
        with open(src, "rb") as f:
            nx, ny, nz, mode = (int(v) for v in np.frombuffer(f.read(16), dtype="<i4"))
        D_stack = nx
    box = min(DOWNSAMPLE_TO or D_stack, D_stack)
    particles, particles_source = src, None
    # identifies the converted stack's origin (for a download: the URL, which survives a runtime reset)
    origin = (f"{PARTICLES.strip()}|{PARTICLES_IN_ARCHIVE}" if str(src).startswith(str(SCRATCH))
              else f"{src}|{src.stat().st_size}|{int(src.stat().st_mtime)}")
    stamp = f"{origin}|{box}"
    if ext not in (".mrcs", ".mrc") or mode != 2 or box < D_stack:
        out = ddir / f"particles.{box}.mrcs"
        meta = ddir / f"particles.{box}.source.txt"
        if not out.exists() or not meta.exists() or meta.read_text() != stamp:
            why = ("converting to a single float32 stack" if box == D_stack else f"downsampling {D_stack} → {box} px")
            print(f"Preparing particles ({why})...")
            tmp = SCRATCH / DATASET_NAME / out.name    # write on fast local disk first, then keep a copy in WORK_DIR
            tmp.parent.mkdir(parents=True, exist_ok=True)
            convert_particles(src, tmp, box, datadir, apix_out=float(ctf_arr[0, 1]) * D_orig / box)
            shutil.copy(tmp, str(out) + ".part") if on_drive(out) else shutil.move(tmp, str(out) + ".part")
            Path(str(out) + ".part").replace(out)
            meta.write_text(stamp)
        particles = out
    elif str(src).startswith(str(SCRATCH)):            # a download that needs no conversion: keep it in WORK_DIR
        out = ddir / src.name
        if not out.exists() or out.stat().st_size != src.stat().st_size:
            shutil.copy(src, str(out) + ".part"); Path(str(out) + ".part").replace(out)
        particles = out
        print(f"Box is already {D_stack} px and float32 - using the stack as is.")
    else:
        print(f"Box is already {D_stack} px and float32 - using the stack as is.")
    if COPY_TO_LOCAL_DISK:
        particles, particles_source = cache_locally(particles, DATASET_NAME)

    # ---- inspect ----
    info = json.loads(hydra_python(f"""
        import json, pickle, numpy as np
        from cryodrgnai import dataset
        imgs = dataset.load_particles({str(particles)!r}, lazy=True)
        N = len(imgs); first = np.stack([imgs[i].get() for i in np.linspace(0, N - 1, 16).astype(int)])
        np.save({str(ddir / 'preview.npy')!r}, first.astype(np.float32))
        print(json.dumps(dict(N=N, D=int(first.shape[-1]))))
    """).strip().splitlines()[-1])
    N, D = info["N"], info["D"]
    apix = float(ctf_arr[0, 1]) * D_orig / D
    assert len(ctf_arr) == N, f"The CTF file has {len(ctf_arr)} entries but there are {N} particles!"
    if pose_pkl is not None:
        n_pose = len(pose[0]) if isinstance(pose, tuple) else len(pose)
        assert n_pose == N, f"The pose file has {n_pose} entries but there are {N} particles!"
    assert D % 2 == 0, f"Box size must be even (got {D})."
    ind = str(_resolve(SUBSET_INDEX_PKL)) if SUBSET_INDEX_PKL.strip() else None
    if MAX_PARTICLES and MAX_PARTICLES < (len(load_pkl(ind)) if ind else N):
        base = np.asarray(load_pkl(ind)) if ind else np.arange(N)
        sub = np.sort(np.random.default_rng(0).choice(base, MAX_PARTICLES, replace=False)).astype(np.int64)
        ind = str(ddir / f"random_subset_{MAX_PARTICLES}_{fingerprint(origin, ind)[:6]}.pkl")
        save_pkl(sub, ind)
        print(f"Using a random subset of {MAX_PARTICLES} of {len(base)} particles.")
    # the particle file 8-a exports subsets from: it must outlive the runtime, so downloads use the WORK_DIR copy
    export_src = src
    if str(src).startswith(str(SCRATCH)):
        if ext in (".star", ".cs"):
            export_src = ddir / src.name
            shutil.copy(src, export_src)
        else:          # a downsampled copy would not match the CTF file's pixel size, so only a full-size one
            export_src = (particles_source or particles) if box == D_orig else None

    st = load_state()
    st["datasets"][DATASET_NAME] = dict(
        particles=str(particles), particles_source=str(particles_source) if particles_source else None,
        ctf=str(ctf_pkl), pose=str(pose_pkl) if pose_pkl else None, labels=None, datadir=None, ind=ind,
        N=N, D=D, apix=apix, has_shifts=True, synthetic=False,
        source_file=str(export_src) if export_src else None, source_datadir=datadir,   # the original file, for exporting particle subsets (8-a)
        fingerprint=fingerprint(stamp, file_hash(ctf_pkl), file_hash(pose_pkl), file_hash(ind)))
    st["active_dataset"] = DATASET_NAME
    save_state(st)
    print(f"✅ Dataset '{DATASET_NAME}': {N} particles, {D}×{D} px, {apix:.3f} Å/px "
          f"(original {D_orig} px @ {ctf_arr[0, 1]:.3f} Å/px), {ctf_arr[0, 5]:.0f} kV, "
          f"defocus {ctf_arr[:, 2].min():.0f}–{ctf_arr[:, 2].max():.0f} Å" + (f", subset: {ind}" if ind else ""))
    mem_gb = 28 * (len(load_pkl(ind)) if ind else N) * D ** 2 / 1e9            # Hydra keeps real + Hartley copies (+ temporaries while loading)
    if mem_gb > 0.5 * psutil.virtual_memory().total / 1e9:
        print(f"⚠️ Loading all particles needs ≈{mem_gb:.1f} GB RAM - enable lazy loading in step 4 or downsample further.")
    montage(np.load(ddir / "preview.npy"), ncols=8, suptitle="Particles (evenly spaced through the stack)")

---
## 4️⃣ Configure the experiment

Pick a **preset**, then override any field you need. Fields left at `0`, `-1` or blank use the preset value or Hydra's default.

| Preset | Pose search | SGD epochs | Network | Use it for |
|:--|:--|:--|:--|:--|
| **Quick** | 2·N images (at most 100 000) = 3 passes over the data, coarse rotation grid (`base_healpy: 1`) | about 40, batch 64 | 3 × 128 | the demo, or a first look at a new dataset |
| **Standard** | 5·N images, kept within 50 000 – 500 000 (6 passes for 10 000 – 100 000 particles, more for smaller datasets) | about 50, batch 64 (256 above 50 000 particles) | 3 × 128 | most real datasets on Colab |
| **Paper-like** | max(20·N, 100 000) images | about 100, batch 64 | 3 × 128 | settings close to the Hydra paper (long; Colab Pro) |

Pose search runs in whole passes over the data (`N_IMGS_POSE_SEARCH // N + 1` of them). The number of SGD epochs is rounded up so that the last epoch is saved as a checkpoint, and the SGD batch size is lowered automatically if it would not fit in GPU memory. All presets use the 128-wide networks of the Hydra paper; Hydra's code default is 256 (`HYPERVOLUME_DIM = 256`), which is roughly 3× slower. The cell prints a rough time estimate; if it is far longer than a Colab session, start with a random subset (`MAX_PARTICLES` in step 3-B).

Training runs in two phases. **Hierarchical pose search** estimates every particle's orientation from scratch; it is the expensive part, and its cost grows with the number of classes *K*. **SGD** then jointly refines poses, class assignments, conformations and the *K* neural fields, and is much cheaper per epoch.

Hydra-specific parameters:
* **`N_CLASSES`** (*K*) is the number of neural fields. Use at least as many as the distinct particle types you expect. Extra classes usually end up empty or collect junk particles.
* **`Z_DIM`** is the dimension of each class's conformational latent space; the paper uses 2. Use `RECONSTRUCTION_TYPE = homo` for rigid classes.
* **`STD_NOISE`** (σ) controls how evenly particles spread over classes: larger values give more uniform class posteriors. The paper used 0.1 for synthetic data and 10 for a cell-lysate dataset; the code default is 1. **`LR_SCORE_TABLE`** is the learning rate of the class scores.

In [ ]:
#@title 4. Configure the experiment { display-mode: "form" }
EXPERIMENT_NAME = "hydra_run1"  #@param {type:"string"}
DATASET = ""  #@param {type:"string"}
#@markdown <small>`DATASET` blank = the active dataset: the one prepared or selected last (step 3 or 8-a). Its name and size are printed below.</small>
PRESET = "Quick (demo / first look)"  #@param ["Quick (demo / first look)", "Standard", "Paper-like (long)"]
#@markdown ### Model
RECONSTRUCTION_TYPE = "het"  #@param ["het", "homo"]
N_CLASSES = 3  #@param {type:"slider", min:1, max:12, step:1}
Z_DIM = 2  #@param {type:"integer"}
POSE_ESTIMATION = "abinit"  #@param ["abinit", "refine", "fixed"]
CONF_ESTIMATION = "autodecoder"  #@param ["autodecoder", "encoder"]
#@markdown ### Training length (0 = preset)
N_IMGS_POSE_SEARCH = 0  #@param {type:"integer"}
EPOCHS_SGD = 0  #@param {type:"integer"}
CHECKPOINT_EVERY = 0  #@param {type:"integer"}
#@markdown <small>`CHECKPOINT_EVERY`: save weights, latents and class maps every N epochs (preset: 10 for Quick, 5 otherwise; pose-search epochs are always saved).</small>
#@markdown ### Advanced (0 / -1 = preset or Hydra default)
STD_NOISE = 0.0  #@param {type:"number"}
#@markdown <small>σ of the class likelihood (Hydra default 1); larger = more uniform class assignments.</small>
LR_SCORE_TABLE = 0.0  #@param {type:"number"}
#@markdown <small>Learning rate of the per-particle class scores (Hydra default 0.01).</small>
BATCH_SIZE_HPS = 0  #@param {type:"integer"}
#@markdown <small>Images per pose-search batch (0 = largest that fits the GPU, at most 8). Smaller is slower, not worse.</small>
BATCH_SIZE_SGD = 0  #@param {type:"integer"}
#@markdown <small>Images per SGD batch (0 = preset: 64, or 256 above 50 000 particles, lowered automatically if it would not fit the GPU).</small>
HYPERVOLUME_DIM = 0  #@param {type:"integer"}
#@markdown <small>Width of each class's neural field (presets: 128; Hydra's code default is 256).</small>
T_EXTENT_PIXELS = -1.0  #@param {type:"number"}
#@markdown <small>Largest particle shift searched, in pixels of the (downsampled) box (Hydra default 20; scale it down with downsampling; 0 for the shift-free synthetic demo, set automatically).</small>
LAZY_LOADING = False  #@param {type:"boolean"}
#@markdown <small>Read particles from disk on the fly instead of loading them all into RAM (slower; needs ≥ 1000 particles).</small>
SEED = -1  #@param {type:"integer"}
LOG_GROUND_TRUTH_LABELS = True  #@param {type:"boolean"}
#@markdown <small>With ground-truth labels (synthetic demo), Hydra colours each class's latent-space PCA in TensorBoard by the true label (needs Z_DIM ≥ 2). Step 7-a computes the accuracy against the ground truth either way.</small>
EXTRA_CONFIG = ""  #@param {type:"string"}
#@markdown <small>Any other Hydra/DRGN-AI parameters as YAML flow mapping, e.g. `l_end: 40, lr: 0.0002, pe_dim: 64`.</small>

import yaml
require_install()
st = load_state()
if EXPERIMENT_NAME in st["experiments"]:
    if training_running(st["experiments"][EXPERIMENT_NAME]):
        raise RuntimeError(f"'{EXPERIMENT_NAME}' is training right now - stop it in 5-c, or choose a new EXPERIMENT_NAME.")
    prev = latest_checkpoint(WORK / "experiments" / EXPERIMENT_NAME)
    if prev:
        display(Markdown(f"⚠️ `{EXPERIMENT_NAME}` already has results (up to epoch {prev[1]}). Training it again (step 5, "
                         "*start over*) moves them to `previous_runs/`, and steps 6–8 then show only the new run. "
                         "Use a new `EXPERIMENT_NAME` (e.g. `hydra_run2`) to keep both side by side."))
if st.get("installed", {}).get("cuda_ok") is False:
    display(Markdown("🛑 Step 2 found that PyTorch cannot use the GPU - training would run on the CPU. Re-run step 2."))
elif not gpu_info():
    display(Markdown("⚠️ No GPU in this runtime: step 5 will only train on the CPU if you tick `ALLOW_CPU_TRAINING` there."))
ds_name, ds = get_dataset(DATASET)
ds = ensure_particles(ds_name)
N = ds["N"] if not ds.get("ind") else len(load_pkl(ds["ind"]))
if ds.get("ind"):
    display(Markdown(f"**Note:** dataset `{ds_name}` is a subset ({N} of {ds['N']} particles)."))

# Pose search dominates the cost: it scales with the number of images searched, with K and with the network size.
# The SGD phase needs enough gradient steps (N / batch_size_sgd per epoch) for the maps to sharpen.
presets = {
    "Quick (demo / first look)": dict(n_imgs_pose_search=min(2 * N, 100_000), epochs_sgd=40,
                                      hypervolume_dim=128, base_healpy=1, batch_size_sgd=64, log_heavy_interval=10),
    "Standard": dict(n_imgs_pose_search=min(max(5 * N, 50_000), 500_000), epochs_sgd=50, hypervolume_dim=128,
                     batch_size_sgd=64 if N <= 50_000 else 256, log_heavy_interval=5),
    "Paper-like (long)": dict(n_imgs_pose_search=max(20 * N, 100_000), epochs_sgd=100, hypervolume_dim=128,
                              batch_size_sgd=64, log_heavy_interval=5),
}[PRESET]

het = RECONSTRUCTION_TYPE == "het"
cfg = {
    "particles": ds["particles"], "ctf": ds["ctf"],
    "quick_config": {"capture_setup": "spa", "reconstruction_type": RECONSTRUCTION_TYPE,
                     "pose_estimation": POSE_ESTIMATION, "conf_estimation": CONF_ESTIMATION if het else None},
    "n_classes": int(N_CLASSES),
    **presets,
}
if ds.get("pose"):
    cfg["pose"] = ds["pose"]
elif POSE_ESTIMATION != "abinit":
    raise ValueError(f"POSE_ESTIMATION={POSE_ESTIMATION!r} needs poses, but dataset '{ds_name}' has none.")
if ds.get("datadir"):
    cfg["datadir"] = ds["datadir"]
if ds.get("relion31"):
    cfg["relion31"] = True
if ds.get("ind"):
    cfg["ind"] = ds["ind"]
if het:
    cfg["z_dim"] = int(Z_DIM)
if ds.get("labels") and LOG_GROUND_TRUTH_LABELS:
    cfg["labels"] = ds["labels"]
if not ds.get("has_shifts", True) and T_EXTENT_PIXELS < 0:
    cfg["t_extent"] = 0.0          # synthetic data without shifts: skip translation search
overrides = dict(n_imgs_pose_search=N_IMGS_POSE_SEARCH, epochs_sgd=EPOCHS_SGD, log_heavy_interval=CHECKPOINT_EVERY,
                 std_noise=STD_NOISE, lr_score_table=LR_SCORE_TABLE, batch_size_hps=BATCH_SIZE_HPS,
                 batch_size_sgd=BATCH_SIZE_SGD, hypervolume_dim=HYPERVOLUME_DIM)
cfg.update({k: v for k, v in overrides.items() if v})
if T_EXTENT_PIXELS >= 0:
    cfg["t_extent"] = float(T_EXTENT_PIXELS)
if LAZY_LOADING:
    cfg["lazy"] = True
if SEED >= 0:
    cfg["seed"] = int(SEED)
extra = {}
if EXTRA_CONFIG.strip():
    extra = yaml.safe_load("{" + EXTRA_CONFIG.strip().strip("{}") + "}")
    for k, v in extra.items():   # YAML reads "1e-4" as a string; Hydra needs a number
        if isinstance(v, str) and re.fullmatch(r"[-+]?(\d+\.?\d*|\.\d+)[eE][-+]?\d+", v):
            extra[k] = float(v)
    cfg.update(extra)
if cfg.get("lazy") and N < 1000:
    raise ValueError("Hydra's lazy loading needs at least 1000 particles; untick LAZY_LOADING.")
if het and CONF_ESTIMATION == "encoder" and int(N_CLASSES) == 1 and not cfg.get("variational_het"):
    raise ValueError("Hydra's encoder mode needs N_CLASSES ≥ 2 (or `variational_het: true` in EXTRA_CONFIG); "
                     "use CONF_ESTIMATION = autodecoder for a single class.")
# Log progress about 10 times per epoch (Hydra's default is every 10 000 images), for the dashboard in step 5.
cfg.setdefault("log_interval", max(256, N // 10))
g = gpu_info()
mem_total = g["total"] / 1024 if g else psutil.virtual_memory().total / 1e9     # GB the model must fit into
K, D, hv = int(N_CLASSES), int(ds["D"]), int(cfg.get("hypervolume_dim", 256))

# Holding all particles in RAM needs ≈ 28 bytes per pixel while loading; beyond ~60% of RAM, read them lazily.
ram_gb = 28 * N * D ** 2 / 1e9
if not cfg.get("lazy") and ram_gb > 0.6 * psutil.virtual_memory().total / 1e9:
    if N >= 1000:
        cfg["lazy"] = True
        print(f"Loading all particles would need ≈{ram_gb:.0f} GB of RAM - switched on lazy loading (reads from disk).")
    else:
        print(f"⚠️ Loading all particles needs ≈{ram_gb:.0f} GB of RAM; downsample further in step 3.")

# SGD-phase memory (measured: ≈ 4.9 KB per pixel inside the box's circle per class for a 128-wide network).
# Unless set explicitly, halve the batch until it fits; more, smaller steps train at least as well.
if not BATCH_SIZE_SGD and "batch_size_sgd" not in extra:
    per_img = 4.9e3 * K * np.pi * (D / 2) ** 2 * hv / 128 / 1e9
    want = int(cfg["batch_size_sgd"])
    cfg["batch_size_sgd"] = next((b for b in (256, 128, 64, 32, 16, 8) if b <= want and 0.5 + per_img * b <= 0.8 * mem_total), 4)
    if cfg["batch_size_sgd"] < want:
        print(f"SGD batch size set to {cfg['batch_size_sgd']} so that {K} classes at {D} px fit in "
              f"{mem_total:.0f} GB of {'GPU' if g else 'CPU'} memory.")

# Pose search memory grows with batch_size_hps × K (measured: ≈ 1.4 GB + 0.4 GB per image per class for a
# 128-wide network). Unless set explicitly, pick the largest batch that fits comfortably on this GPU.
if "batch_size_hps" not in cfg:
    per_image = 0.41 * K * hv / 128
    cfg["batch_size_hps"] = next((b for b in (8, 4, 2) if 1.4 + per_image * b <= 0.85 * mem_total), 1)
    if cfg["batch_size_hps"] < 8:
        print(f"Pose-search batch size set to {cfg['batch_size_hps']} so that {K} classes fit in "
              f"{mem_total:.0f} GB of {'GPU' if g else 'CPU'} memory.")

# Hydra only writes checkpoints every `log_heavy_interval` epochs (plus every pose-search epoch), and not
# necessarily at the very end; extend the SGD phase slightly so that the final epoch is saved.
n_ps_epochs = 0 if POSE_ESTIMATION != "abinit" else max(2, cfg["n_imgs_pose_search"] // N + 1)
if POSE_ESTIMATION != "abinit":
    cfg["n_imgs_pose_search"] = 0
every = int(cfg.get("log_heavy_interval", 5))
last = n_ps_epochs + cfg["epochs_sgd"] - 1
if every > 0 and last % every:
    cfg["epochs_sgd"] += every - last % every
total_epochs = n_ps_epochs + cfg["epochs_sgd"]

# Rough T4 timing, to catch runs far longer than a Colab session: pose search manages ≈ 6 image-class searches
# per second (more with l_end lowered), and the SGD phase ≈ 1.5 TFLOP/s through K networks at every pixel.
t_ps = n_ps_epochs * N * K / 6.0
flop_img = 6 * np.pi * (D / 2) ** 2 * K * (2 * int(cfg.get("pe_dim", 64)) * hv + 3 * hv ** 2)
t_sgd = (cfg["epochs_sgd"] + 1) * N * flop_img / 1.5e12 + (cfg["epochs_sgd"] + 1) * N / cfg["batch_size_sgd"] * 0.02
t_est = t_ps + t_sgd

exp_dir = WORK / "experiments" / EXPERIMENT_NAME
exp_dir.mkdir(parents=True, exist_ok=True)
cfg_file = exp_dir / "configs.yaml"
if cfg_file.exists():
    shutil.copy(cfg_file, exp_dir / "configs.previous.yaml")
with open(cfg_file, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# `drgnai setup` validates the configuration (and fills in absolute paths)
run(f"{q(str(DRGNAI))} setup {q(str(exp_dir))}", quiet=True)

# What a checkpoint depends on: the model and the data, not batch sizes, logging or the length of the SGD phase.
# Training refuses to resume across a change of these (step 5).
RUNTIME_KEYS = {"batch_size_hps", "batch_size_sgd", "batch_size_known_poses", "lazy", "num_workers", "max_threads",
                "log_interval", "log_heavy_interval", "epochs_sgd", "verbose_time", "load"}
signature = json.dumps([{k: v for k, v in cfg.items() if k not in RUNTIME_KEYS}, ds.get("fingerprint")],
                       sort_keys=True, default=str)
st = load_state()
st["experiments"][EXPERIMENT_NAME] = dict(dir=str(exp_dir), dataset=ds_name, dataset_record=ds, n_particles=N,
                                          n_classes=int(N_CLASSES), z_dim=cfg.get("z_dim", 0),
                                          total_epochs=total_epochs, pose_search_epochs=n_ps_epochs,
                                          epochs_sgd=cfg["epochs_sgd"], n_imgs_pose_search=cfg["n_imgs_pose_search"],
                                          config_signature=signature,
                                          trained_signature=st["experiments"].get(EXPERIMENT_NAME, {}).get("trained_signature"))
st["active_experiment"] = EXPERIMENT_NAME
save_state(st)

display(Markdown(f"**Experiment `{EXPERIMENT_NAME}`** on dataset `{ds_name}` ({N} particles, {ds['D']} px) → `{exp_dir}`  \n"
                 f"{n_ps_epochs} pose-search epoch(s) + {cfg['epochs_sgd']} SGD epochs (+ 1 short pretraining epoch); "
                 f"checkpoints every {cfg.get('log_heavy_interval', 5)} epochs and after every pose-search epoch.  \n"
                 f"Rough time on a T4: **≈ {fmt_rough(t_est)}** (pose search {fmt_rough(t_ps)}, SGD {fmt_rough(t_sgd)}); "
                 "L4/A100 GPUs are roughly 2–5× faster. Step 5 shows a measured estimate once training runs."))
if t_est > 6 * 3600:
    display(Markdown("⚠️ That is longer than a free Colab session usually lasts. Training resumes from the last checkpoint "
                     "(step 5, `START = auto`), but consider a random subset (`MAX_PARTICLES` in step 3-B), fewer classes, "
                     "the Quick preset, or a lower `l_end` (e.g. `l_end: 24` in `EXTRA_CONFIG`) for a first look."))
print(cfg_file.read_text())

---
## 5️⃣ Train

Training runs as a **separate background process**. The cell shows a live dashboard (phase, epoch, progress within the epoch, loss, time left, GPU use) until training finishes. While it is shown, other cells wait in the queue. Press ■ to stop *watching* and use the notebook; training continues. Re-attach with **5-b**, or stop training with **5-c**.

> ⚠️ **Keep this tab open and a dashboard cell running.** Colab shuts down runtimes that sit idle (often after about 90 minutes on the free tier) and limits session length; either ends training. For long runs, put `WORK_DIR` on Google Drive (step 1) so you can resume.

Outputs are written to `WORK_DIR/experiments/<name>/out/`:

| File | Content |
|:--|:--|
| `weights.<E>.pkl` | model checkpoint at epoch *E* (used for analysis / resuming) |
| `p_classes.<E>.pkl` | per-particle class probabilities (N × K) |
| `conf.<E>.pkl` | per-particle conformational latent vector (N × z_dim) |
| `pose.<E>.pkl` | per-particle rotations and translations (in pixels) for the best class |
| `reconstruct.epoch_<E>.class_<k>.mrc` | one map per non-empty class, at one particle's conformation (homogeneous runs: class 0 only; step 6 makes maps for all classes) |
| `training.log`, `summaries/` | text log and TensorBoard event files |

> **Resuming.** If the runtime disconnects (keep results on Google Drive!), re-run steps 1 and 2, then this cell: with `START = auto` it continues from the newest checkpoint and finishes the original schedule. The interrupted run's files are kept in a timestamped `out.<time>` folder.

In [ ]:
#@title 5. Train { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
#@markdown <small>Blank = the experiment configured last in step 4.</small>
START = "auto (resume if possible)"  #@param ["auto (resume if possible)", "resume", "start over"]
#@markdown <small>`auto`: continue from the newest checkpoint if an earlier run of this experiment was interrupted, otherwise start a new run. `start over` moves earlier results to `previous_runs/` and trains from scratch.</small>
WATCH = True  #@param {type:"boolean"}
REFRESH_SECONDS = 20  #@param {type:"integer"}
ALLOW_CPU_TRAINING = False  #@param {type:"boolean"}
#@markdown <small>Without a GPU, training refuses to start unless this is ticked: on a CPU even the demo takes about a day.</small>

require_install()
import yaml
exp_name, exp = get_experiment(EXPERIMENT)
exp_dir = Path(exp["dir"]); out_dir = exp_dir / "out"
launch = True
if training_running(exp):
    print(f"Training of '{exp_name}' is already running - attaching to it.")
    launch = False
else:
    if not ALLOW_CPU_TRAINING and (not gpu_info() or load_state().get("installed", {}).get("cuda_ok") is False):
        raise RuntimeError("No usable GPU in this runtime. Switch to a GPU (Runtime ▸ Change runtime type ▸ T4 GPU, then "
                           "re-run steps 1–2), or tick ALLOW_CPU_TRAINING to train on the CPU anyway (very slow).")
    cfg = yaml.safe_load((exp_dir / "configs.yaml").read_text())
    now_fp = load_state()["datasets"].get(exp["dataset"], {}).get("fingerprint")
    if exp.get("dataset_record") and now_fp != exp["dataset_record"].get("fingerprint"):
        raise RuntimeError(f"Dataset '{exp['dataset']}' was prepared again (step 3) after this experiment was configured. "
                           "Re-run step 4 so that the experiment uses the new version.")
    ensure_particles(exp["dataset"])
    if not Path(cfg["particles"]).exists():
        raise FileNotFoundError(f"Particle stack {cfg['particles']} is missing - re-run step 3 for this dataset.")
    cfg.pop("load", None)                                    # restore the full schedule chosen in step 4
    cfg["epochs_sgd"] = exp["epochs_sgd"]
    cfg["n_imgs_pose_search"] = exp.get("n_imgs_pose_search", cfg.get("n_imgs_pose_search", 0))
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    has_output = out_dir.exists() and any(out_dir.iterdir())
    final = exp["total_epochs"] - 1                          # index of the last epoch of the schedule
    n_ps = int(exp.get("pose_search_epochs", 0))
    ckpt = latest_checkpoint(exp_dir)
    changed = ckpt is not None and exp.get("trained_signature") not in (None, exp.get("config_signature"))
    if changed and START.startswith("auto"):
        print("The configuration changed in step 4 since the existing checkpoints were made - starting a new run.")
    if changed and START.startswith("resume"):
        raise RuntimeError("The configuration changed in step 4 since the checkpoints were made; resuming would mix "
                           "two set-ups. Use START = 'start over' (or restore the old settings in step 4).")
    resume = START.startswith("resume") or (START.startswith("auto") and ckpt is not None and not changed)
    if START.startswith("resume") and ckpt is None:
        raise RuntimeError("This experiment has no checkpoint (weights.*.pkl) to resume from yet.")
    if resume and ckpt[1] >= final:
        print(f"Training of '{exp_name}' already finished (epoch {ckpt[1]} of {final}). Continue with step 6, "
              "or choose START = 'start over' to train it again.")
        launch = resume = False
    if resume:
        if n_ps >= 2 and ckpt[1] == n_ps - 1:
            # Hydra fills the per-particle pose table only when it leaves pose search; resuming from the last
            # pose-search checkpoint would start SGD from blank poses, so repeat that epoch instead.
            prev = next((d / f"weights.{n_ps - 2}.pkl" for d in output_dirs(exp_dir)
                         if (d / f"weights.{n_ps - 2}.pkl").exists()), None)
            if prev is None:
                raise RuntimeError("Cannot resume safely from the last pose-search epoch; use START = 'start over'.")
            print(f"Repeating the last pose-search epoch ({n_ps - 1}) so that the pose table is set up correctly.")
            ckpt = (prev, n_ps - 2)
        if has_output:                                       # keep it next to the new run (same lineage)
            out_dir.rename(exp_dir / f"out.{stamp}")
            if ckpt[0].parent == out_dir:
                ckpt = (exp_dir / f"out.{stamp}" / ckpt[0].name, ckpt[1])
        if ckpt[1] >= n_ps:      # pose search is over: run only SGD epochs, poses come from the checkpoint
            cfg["n_imgs_pose_search"], cfg["epochs_sgd"] = 0, final - ckpt[1]
        else:                    # still inside pose search: Hydra continues it by absolute epoch number
            es = exp["epochs_sgd"] - 1 - ckpt[1]
            if es < 1:           # Hydra needs >= 1 SGD epoch here; grow the schedule (keeping the last epoch saved)
                every = int(cfg.get("log_heavy_interval", 5)) or 1
                es = 1
                while (n_ps + es + ckpt[1]) % every:
                    es += 1
                final = n_ps + es + ckpt[1]
                st = load_state(); st["experiments"][exp_name]["total_epochs"] = final + 1; save_state(st)
                print(f"(Hydra cannot resume pose search this late in a short schedule; it now ends at epoch {final}.)")
            cfg["epochs_sgd"] = es
        cfg["load"] = str(ckpt[0])
        print(f"Resuming from {ckpt[0]} (epoch {ckpt[1]}); {final - ckpt[1]} more epoch(s) to go.")
    elif launch and (has_output or list(exp_dir.glob("out.*"))):   # fresh start: archive earlier runs
        archive = exp_dir / "previous_runs" / f"run.{stamp}"
        archive.mkdir(parents=True)
        for d in [*exp_dir.glob("out.*"), *([out_dir] if has_output else [])]:
            d.rename(archive / d.name)
        print(f"Earlier output of this experiment moved to {archive}")
if launch:
    with open(exp_dir / "configs.yaml", "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    out_dir.mkdir(exist_ok=True)
    exit_file = exp_dir / "train_exit_code"
    exit_file.unlink(missing_ok=True)
    cmd = f"{q(str(DRGNAI))} train {q(str(exp_dir))} --no-analysis; echo $? > {q(str(exit_file))}"
    with open(exp_dir / "train_console.log", "w") as console:
        proc = subprocess.Popen(["bash", "-c", cmd], cwd=str(exp_dir), env=hydra_env(), stdout=console,
                                stderr=subprocess.STDOUT, start_new_session=True)
    _TRAIN_PROCS[exp["dir"]] = proc
    st = load_state()
    st["experiments"][exp_name].update(pid=proc.pid, started=time.time())
    if not resume:      # the checkpoints now belong to this configuration (a resume keeps the original one)
        st["experiments"][exp_name]["trained_signature"] = st["experiments"][exp_name].get("config_signature")
    st["experiments"][exp_name].pop("analyzed_epoch", None)
    save_state(st)
    print(f"🚀 Started training '{exp_name}' (pid {proc.pid}). Console log: {exp_dir / 'train_console.log'}")
if WATCH and (launch or training_running(get_experiment(exp_name)[1])):
    monitor(exp_name, refresh=REFRESH_SECONDS)

In [ ]:
#@title 5-b. Re-attach the live dashboard { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
REFRESH_SECONDS = 20  #@param {type:"integer"}
monitor(EXPERIMENT, refresh=REFRESH_SECONDS)

In [ ]:
#@title 5-c. Stop training { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
CONFIRM_STOP = False  #@param {type:"boolean"}
exp_name, exp = get_experiment(EXPERIMENT)
if not CONFIRM_STOP:
    print("Tick CONFIRM_STOP to stop the training process.")
elif not training_running(exp):
    print(f"'{exp_name}' is not running.")
else:
    os.killpg(os.getpgid(exp["pid"]), signal.SIGTERM)
    time.sleep(3)
    print(f"Sent SIGTERM to training of '{exp_name}'. Checkpoints written so far are kept; resume later with step 5 (START = auto).")

In [ ]:
#@title 5-d. (optional) TensorBoard { display-mode: "form" }
#@markdown Interactive loss curves, class/latent summaries and image reconstructions written by Hydra. Each output folder is its own run: `out/summaries` is the current one, and after a resume the earlier epochs are under `out.<time>/summaries` (runs that were started over: `previous_runs/`).
EXPERIMENT = ""  #@param {type:"string"}
exp_name, exp = get_experiment(EXPERIMENT)
logdir = Path(exp["dir"])
get_ipython().run_line_magic("load_ext", "tensorboard")
get_ipython().run_line_magic("tensorboard", f"--logdir {q(str(logdir))}")

---
## 6️⃣ Analyze

`drgnai analyze` loads a checkpoint and, **for each class**, takes the particles assigned to that class and runs PCA (UMAP when `Z_DIM > 2`) and *k*-means on their conformational latents. It then generates volumes at the *k*-means centres and along the principal components. Results go to `out/analysis.epoch_<E>.class_<k>/`:

* `kmeans<K>/vol_###.mrc` are volumes at the *k*-means centres (+ `labels.pkl`, `centers.txt`, plots); *k*-means indices count only the particles of that class
* `pc1_10/`, `pc2_10/` are volume trajectories along PC1/PC2 from the 5th to the 95th percentile (for `Z_DIM = 1`: `vol_000…009.mrc` along *z* directly in the class folder)
* `z_pca*.png` are latent-space plots; `umap*.png` and `umap.pkl` appear only when `Z_DIM > 2`
* `cryoDRGN-analysis.ipynb` is cryoDRGN's generic template; it expects cryoDRGN's folder names and a single class, so it needs editing before it works on Hydra output

For homogeneous runs (`z_dim = 0`), analysis writes one map per class (`vol_000.mrc`).

In [ ]:
#@title 6. Analyze a checkpoint { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
EPOCH = -1  #@param {type:"integer"}
#@markdown <small>-1 = newest checkpoint.</small>
CLASSES = "all"  #@param {type:"string"}
#@markdown <small>`all` or a comma-separated list such as `0,2`. Classes with fewer than `MIN_PARTICLES` particles are skipped.</small>
MIN_PARTICLES = 50  #@param {type:"integer"}
SKIP_UMAP = False  #@param {type:"boolean"}
#@markdown <small>UMAP is only computed when Z_DIM > 2; it can take several minutes per class on large datasets.</small>
N_PCS = 2  #@param {type:"integer"}
VOLUMES_PER_PC = 10  #@param {type:"integer"}
KMEANS_K = 20  #@param {type:"integer"}
ANALYZE_WHILE_TRAINING = False  #@param {type:"boolean"}
#@markdown <small>Analysing a checkpoint while training is still running uses the CPU (so that training keeps the GPU to itself) and is slower.</small>

import yaml
require_install()
exp_name, exp = get_experiment(EXPERIMENT)
exp_dir = Path(exp["dir"]); out_dir = exp_dir / "out"
env = hydra_env()
if training_running(exp):
    if not ANALYZE_WHILE_TRAINING:
        raise RuntimeError("Training is still running. Wait for it to finish (5-b), or tick ANALYZE_WHILE_TRAINING "
                           "to analyse the newest checkpoint written so far.")
    print("⚠️ Training is still running: analysing the newest checkpoint so far, on the CPU.")
    env["CUDA_VISIBLE_DEVICES"] = ""
epoch = (latest_checkpoint(exp_dir) or (None, None))[1] if EPOCH < 0 else EPOCH
if epoch is not None:
    out_dir = epoch_dir(exp, epoch)
if epoch is None or not (out_dir / f"weights.{epoch}.pkl").exists():
    raise RuntimeError(f"No checkpoint {'yet' if epoch is None else f'weights.{epoch}.pkl'} in {out_dir}. "
                       "Train first (step 5); checkpoints appear after each pose-search epoch.")
analyze_dir = exp_dir
if out_dir != exp_dir / "out":
    # `drgnai analyze` only reads <dir>/out: point a stand-in folder (on local disk; Drive has no symlinks) at it
    analyze_dir = SCRATCH / "analyze_views" / exp_name / out_dir.name
    analyze_dir.mkdir(parents=True, exist_ok=True)
    (analyze_dir / "out").unlink(missing_ok=True)
    (analyze_dir / "out").symlink_to(out_dir)
    print(f"(Epoch {epoch} is in {out_dir.name}/, from before training was resumed.)")
tcfg = yaml.safe_load((out_dir / "train-configs.yaml").read_text())
K, zdim = int(tcfg.get("n_classes", 1)), int(tcfg.get("z_dim", 0))
if K > 1:
    pred = load_pkl(out_dir / f"p_classes.{epoch}.pkl").argmax(1)
    counts = np.bincount(pred, minlength=K)
else:
    counts = np.array([exp["n_particles"]])
wanted = range(K) if CLASSES.strip().lower() == "all" else [int(c) for c in CLASSES.split(",") if c.strip()]
todo = [k for k in wanted if counts[k] >= (1 if zdim == 0 else MIN_PARTICLES)]
print(f"Epoch {epoch}: particles per class = {dict(enumerate(counts.tolist()))}")
print(f"Analysing classes {todo}" + (f" (skipping {sorted(set(wanted) - set(todo))}: < {MIN_PARTICLES} particles)"
                                      if set(wanted) - set(todo) else ""))
t0 = time.time()
for k in todo:
    print(f"\n━━━━━━━━━━ class {k} ({counts[k]} particles) ━━━━━━━━━━")
    ksample = max(1, min(KMEANS_K, int(counts[k])))
    args = f"--epoch {epoch} --class-idx {k} --pc {max(1, min(N_PCS, zdim))} --n-per-pc {VOLUMES_PER_PC} --ksample {ksample}"
    if SKIP_UMAP:
        args += " --skip-umap"
    run(f"{q(str(DRGNAI))} analyze {q(str(analyze_dir))} {args}", quiet=True, env=env)
    print(f"✔ class {k} → {out_dir / f'analysis.epoch_{epoch}.class_{k}'}")
st = load_state(); st["experiments"][exp_name]["analyzed_epoch"] = epoch; save_state(st)
print(f"\n✅ Analysis done in {fmt_dt(time.time() - t0)}. Explore the results in step 7.")

---
## 7️⃣ Explore the results

These cells read Hydra's outputs and plot them directly in the notebook. They run on Colab's own Python and do not need the Hydra environment.

In [ ]:
#@title 7-a. Training overview: loss, class populations, accuracy vs ground truth { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
EPOCH = -1  #@param {type:"integer"}

import yaml
exp_name, exp = get_experiment(EXPERIMENT)
epoch = pick_epoch(exp, EPOCH)
if epoch is None:
    raise RuntimeError("No checkpoint yet - train first (step 5).")
out_dir = epoch_dir(exp, epoch)
tcfg = yaml.safe_load((out_dir / "train-configs.yaml").read_text())
K = int(tcfg.get("n_classes", 1))
ds = exp_dataset(exp)
info = training_info(Path(exp["dir"]))

fig, axes = plt.subplots(1, 3 if K > 1 else 1, figsize=(16 if K > 1 else 6, 3.6), squeeze=False)
axes = axes[0]
done = sorted(e for e in info["loss"] if e >= 0)
if done:
    axes[0].plot(done, [info["loss"][e] for e in done], "o-", ms=3)
    ps = exp.get("pose_search_epochs", 0)
    if ps:
        axes[0].axvspan(-0.5, ps - 0.5, color="orange", alpha=.15, label="pose search")
        axes[0].legend()
    axes[0].set(xlabel="epoch", ylabel="total loss", title="Training loss"); axes[0].grid(alpha=.3)

if K > 1:
    p = load_pkl(out_dir / f"p_classes.{epoch}.pkl")
    pred = p.argmax(1)
    counts = np.bincount(pred, minlength=K)
    axes[1].bar(range(K), counts, color=plt.cm.tab10(np.arange(K) % 10))
    axes[1].set(xlabel="class", ylabel="# particles", title=f"Class populations (epoch {epoch})", xticks=range(K))
    # populations over epochs
    hist = []
    for e, f in sorted(lineage_files(exp["dir"], "p_classes").items()):
        if e < 0:   # the pretraining epoch only sees a subset of the particles
            continue
        hist.append((e, np.bincount(load_pkl(f).argmax(1), minlength=K) / len(pred)))
    if hist:
        es = [h[0] for h in hist]; fr = np.array([h[1] for h in hist])
        for k in range(K):
            axes[2].plot(es, fr[:, k], "o-", ms=3, color=plt.cm.tab10(k % 10), label=f"class {k}")
        axes[2].set(xlabel="epoch", ylabel="fraction of particles", title="Class populations during training")
        axes[2].legend(fontsize=7, ncol=2); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

if K > 1:
    conf = p.max(1)
    print(f"Mean assignment confidence (max class probability): {conf.mean():.3f}; "
          f"{(conf > 0.9).mean() * 100:.1f}% of particles above 0.9")
    if ds.get("labels"):
        from scipy.optimize import linear_sum_assignment
        from sklearn.metrics import adjusted_rand_score
        gt = load_pkl(ds["labels"])
        if tcfg.get("ind") is not None:
            gt = gt[load_pkl(tcfg["ind"])] if isinstance(tcfg["ind"], str) else gt[:int(tcfg["ind"])]
        G = int(gt.max()) + 1
        cm = np.zeros((G, K), int)
        np.add.at(cm, (gt, pred), 1)
        r, c = linear_sum_assignment(-cm)
        acc = cm[r, c].sum() / cm.sum()
        names = ds.get("class_names") or [f"GT {g}" for g in range(G)]
        fig, ax = plt.subplots(figsize=(1.0 + 0.7 * K, 0.8 + 0.6 * G))
        im = ax.imshow(cm / cm.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
        for i in range(G):
            for j in range(K):
                ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                        color="white" if cm[i, j] > 0.6 * cm[i].sum() else "black")
        ax.set(xticks=range(K), yticks=range(G), yticklabels=names, xlabel="predicted class", ylabel="ground truth",
               title=f"Accuracy {acc * 100:.1f}% (best matching) · ARI {adjusted_rand_score(gt, pred):.3f}")
        plt.colorbar(im, ax=ax, fraction=.046); plt.show()
        display(Markdown("Ground truth → Hydra class: " + ", ".join(f"**{names[i]}** → class {j}" for i, j in zip(r, c))))

In [ ]:
#@title 7-b. Latent space of each class { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
EPOCH = -1  #@param {type:"integer"}
COLOR_BY = "ground-truth conformation (synthetic)"  #@param ["ground-truth conformation (synthetic)", "ground-truth class (synthetic)", "k-means cluster", "density"]

import yaml
from sklearn.decomposition import PCA
exp_name, exp = get_experiment(EXPERIMENT)
epoch = pick_epoch(exp, EPOCH)
out_dir = epoch_dir(exp, epoch)
if epoch is None:
    raise RuntimeError("No checkpoint yet - train first (step 5).")
tcfg = yaml.safe_load((out_dir / "train-configs.yaml").read_text())
K, zdim = int(tcfg.get("n_classes", 1)), int(tcfg.get("z_dim", 0))
ds = exp_dataset(exp)
if zdim == 0:
    print("Homogeneous reconstruction (z_dim = 0): there is no conformational latent space to show.")
else:
    z = load_pkl(out_dir / f"conf.{epoch}.pkl")
    pred = load_pkl(out_dir / f"p_classes.{epoch}.pkl").argmax(1) if K > 1 else np.zeros(len(z), int)
    sel = None
    if tcfg.get("ind") is not None:
        sel = load_pkl(tcfg["ind"]) if isinstance(tcfg["ind"], str) else np.arange(int(tcfg["ind"]))
    def _gt(key):
        if not ds.get(key):
            return None
        a = load_pkl(ds[key])
        return a[sel] if sel is not None else a
    gt_conf, gt_lab = _gt("gt_conformation"), _gt("labels")
    classes = [k for k in range(K) if (pred == k).sum() >= 10]
    adirs = {k: out_dir / f"analysis.epoch_{epoch}.class_{k}" for k in classes}
    has_umap = any((adirs[k] / "umap.pkl").exists() for k in classes)
    if not classes:
        raise RuntimeError("No class has at least 10 particles at this epoch.")
    if COLOR_BY.startswith("ground-truth") and gt_conf is None:
        print("This dataset has no ground truth; colouring by density instead.")
    fig, axes = plt.subplots(2 if has_umap else 1, len(classes), figsize=(4.2 * len(classes), 8 if has_umap else 4),
                             squeeze=False)
    for col, k in enumerate(classes):
        m = pred == k
        zk = z[m]
        pc = PCA(n_components=2).fit_transform(zk) if zdim > 1 else np.c_[zk[:, 0], np.random.default_rng(0).normal(0, .05, m.sum())]
        emb = [("PCA" if zdim > 1 else "z (1D, jittered)", pc)]
        if has_umap:
            umap_f = adirs[k] / "umap.pkl"
            emb.append(("UMAP", load_pkl(umap_f) if umap_f.exists() else None))
        kfile = newest(adirs[k].glob("kmeans*/labels.pkl"))
        for row, (title, e) in enumerate(emb):
            ax = axes[row, col]
            if e is None:
                ax.axis("off"); ax.set_title(f"class {k}: no UMAP (re-run step 6 without SKIP_UMAP)", fontsize=9); continue
            c, cmap = None, None
            if COLOR_BY.startswith("ground-truth conformation") and gt_conf is not None:
                c, cmap = gt_conf[m], "viridis"
            elif COLOR_BY.startswith("ground-truth class") and gt_lab is not None:
                c, cmap = gt_lab[m], "tab10"
            elif COLOR_BY.startswith("k-means") and kfile:
                c, cmap = load_pkl(kfile), "tab20"
            if c is None:
                ax.hexbin(e[:, 0], e[:, 1], gridsize=40, cmap="Blues", mincnt=1)
            else:
                sc = ax.scatter(e[:, 0], e[:, 1], c=c, cmap=cmap, s=3, alpha=.6, rasterized=True)
                if cmap == "viridis":
                    plt.colorbar(sc, ax=ax, fraction=.046, label="true hinge angle (0-1)")
                elif cmap == "tab20":       # label each k-means cluster at its median position (used by 8-a)
                    for cl in np.unique(c):
                        mx, my = np.median(e[c == cl], axis=0)
                        ax.text(mx, my, str(cl), fontsize=8, weight="bold", ha="center", va="center",
                                bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=.7, lw=0))
            ax.set_title(f"class {k} · {title} ({m.sum()} particles)", fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
    if not has_umap and zdim == 2:
        print("UMAP plots appear here when Z_DIM > 2 (Hydra only computes UMAP then).")
    if COLOR_BY.startswith("k-means") and not any(next(adirs[k].glob("kmeans*/labels.pkl"), None) for k in classes):
        print(f"No k-means labels for epoch {epoch}: run step 6 for this epoch (with Z_DIM = 1 Hydra does not save them).")
    fig.suptitle(f"Epoch {epoch}", y=1.0)
    plt.tight_layout(); plt.show()
    if gt_conf is not None and COLOR_BY.startswith("ground-truth conformation"):
        from scipy.stats import spearmanr
        for k in classes:
            m = pred == k
            if m.sum() > 20 and zdim > 0:
                pc1 = PCA(n_components=1).fit_transform(z[m])[:, 0]
                print(f"class {k}: |Spearman ρ| between latent {'z' if zdim == 1 else 'PC1'} and true hinge angle = {abs(spearmanr(pc1, gt_conf[m])[0]):.2f}")

In [ ]:
#@title 7-c. Volume gallery: one map per class, and conformations along PC1 { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
EPOCH = -1  #@param {type:"integer"}
SHOW_GROUND_TRUTH = True  #@param {type:"boolean"}

import yaml
exp_name, exp = get_experiment(EXPERIMENT)
epoch = pick_epoch(exp, EPOCH)
out_dir = epoch_dir(exp, epoch)
if epoch is None:
    raise RuntimeError("No checkpoint yet - train first (step 5).")
ds = exp_dataset(exp)
tcfg = yaml.safe_load((out_dir / "train-configs.yaml").read_text())
K, zdim = int(tcfg.get("n_classes", 1)), int(tcfg.get("z_dim", 0))


def _show_vols(files, labels, title):
    if not files:
        return
    imgs, titles = [], []
    for f, lab0 in zip(files, labels):
        for a, lab in zip(projections(oriented(read_mrc(f)[0])), ("along z", "along y", "along x")):
            imgs.append(a); titles.append(f"{lab0} · {lab}")
    montage(imgs, ncols=6 if len(files) > 1 else 3, titles=titles, size=1.9, cmap="magma", suptitle=title)


def _trajectory(adir):
    """Volumes along the main axis of variation of one class (from step 6)."""
    if zdim == 1:
        return sorted(adir.glob("vol_*.mrc")), "latent z, 5th → 95th percentile"
    pcdirs = sorted(adir.glob("pc1_*"))
    return (sorted(pcdirs[0].glob("vol_*.mrc")) if pcdirs else []), "latent PC1, 5th → 95th percentile"


# one map per class: from the analysis (step 6) where available, else Hydra's checkpoint map
files, labels, missing = [], [], []
for k in range(K):
    adir = out_dir / f"analysis.epoch_{epoch}.class_{k}"
    traj = _trajectory(adir)[0] if zdim else []
    if zdim == 0 and (adir / "vol_000.mrc").exists():
        files.append(adir / "vol_000.mrc"); labels.append(f"class {k}")
    elif traj:
        files.append(traj[len(traj) // 2]); labels.append(f"class {k} (median)")
    elif (out_dir / f"reconstruct.epoch_{epoch}.class_{k}.mrc").exists():
        files.append(out_dir / f"reconstruct.epoch_{epoch}.class_{k}.mrc")
        labels.append(f"class {k}" + (" (1 particle's z)" if zdim else ""))
    else:
        missing.append(k)
_show_vols(files, labels, f"Hydra class maps at epoch {epoch} (projections)")
if missing:
    print(f"No map for classes {missing} at epoch {epoch}: they are empty, or (homogeneous runs) not analysed yet - run step 6.")

if zdim:
    for k in range(K):
        traj, what = _trajectory(out_dir / f"analysis.epoch_{epoch}.class_{k}")
        if traj:
            vols = [oriented(read_mrc(f)[0]) for f in traj]
            montage([projections(v)[0] for v in vols], ncols=len(vols), titles=[f"{i}" for i in range(len(vols))],
                    size=1.7, cmap="magma", suptitle=f"class {k}, epoch {epoch}: volumes along {what} (projection along z)")
    if not any(_trajectory(out_dir / f"analysis.epoch_{epoch}.class_{k}")[0] for k in range(K)):
        print(f"Run step 6 for epoch {epoch} to see each class's conformational trajectory here.")

if SHOW_GROUND_TRUTH and ds.get("gt_volumes"):
    gt = sorted(Path(ds["gt_volumes"]).glob("*conf0.0.mrc"))
    _show_vols(gt, [re.sub(r"^class\d+_|_conf.*$", "", f.stem).replace("_", " ") for f in gt],
               "Ground truth (conformation 0) for comparison - ab initio maps can be rotated or mirrored relative to it")

In [ ]:
#@title 7-d. Interactive 3D viewer { display-mode: "form" }
#@markdown Pick any `.mrc` produced by the experiment (or the ground truth) and drag the threshold slider.
EXPERIMENT = ""  #@param {type:"string"}
import ipywidgets as widgets
import plotly.graph_objects as go
from skimage import measure

exp_name, exp = get_experiment(EXPERIMENT)
exp_dir = Path(exp["dir"])
ds = exp_dataset(exp)
files = sorted(exp_dir.glob("out*/reconstruct.*.mrc"), key=lambda f: f.stat().st_mtime, reverse=True)
files += sorted(exp_dir.glob("out*/analysis.*/**/*.mrc"))
if ds.get("gt_volumes"):
    files += sorted(Path(ds["gt_volumes"]).glob("*.mrc"))
labels = {str(f.relative_to(exp_dir)) if exp_dir in f.parents else f"ground truth/{f.name}": f for f in files}
if not labels:
    print("No volumes yet - train (step 5) and analyse (step 6) first.")
else:
    _cache = {}
    def _vol(key):
        if key not in _cache:
            _cache[key] = oriented(read_mrc(labels[key])[0])
        return _cache[key]
    def show(volume, threshold_sigma, colour):
        v = _vol(volume)
        level = float(v.mean() + threshold_sigma * v.std())
        if not v.min() < level < v.max():
            print("Threshold outside the density range."); return
        verts, faces, _, _ = measure.marching_cubes(v, level, step_size=1 if v.shape[0] <= 128 else 2)
        fig = go.Figure(go.Mesh3d(x=verts[:, 2], y=verts[:, 1], z=verts[:, 0], i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
                                  color=colour, opacity=1.0, flatshading=False,
                                  lighting=dict(ambient=.35, diffuse=.8, specular=.3, roughness=.6)))
        fig.update_layout(height=560, margin=dict(l=0, r=0, t=30, b=0), title=f"{volume}  (level = mean + {threshold_sigma:.1f}σ)",
                          scene=dict(aspectmode="data", xaxis_visible=False, yaxis_visible=False, zaxis_visible=False))
        fig.show()
    widgets.interact(show,
                     volume=widgets.Dropdown(options=list(labels), layout=widgets.Layout(width="70%")),
                     threshold_sigma=widgets.FloatSlider(value=2.5, min=0.5, max=8, step=0.25, continuous_update=False),
                     colour=widgets.Dropdown(options=["lightblue", "gold", "salmon", "lightgreen", "plum"]))

---
## 8️⃣ Filter particles and export results

Hydra separates particles into classes but does not produce half-maps or an FSC. A common next step is to take the particles of one class back to RELION or cryoSPARC for high-resolution refinement. **8-a** writes the selection as an index `.pkl` and as a `.star`/`.cs` that refers to your original particle files, so the same particles are used at full resolution.

In [ ]:
#@title 8-a. Select particles by class or k-means cluster (Colab-friendly replacement for `drgnai filter`) { display-mode: "form" }
#@markdown Writes an index `.pkl` (indices into your original particle file), exports the selection as a `.star`/`.cs` for RELION or cryoSPARC, and registers a new dataset so you can re-run steps 4–7 on the subset (e.g. to drop a junk class or study one class alone).
SELECT_PARTICLES = False  #@param {type:"boolean"}
EXPERIMENT = ""  #@param {type:"string"}
EPOCH = -1  #@param {type:"integer"}
KEEP_CLASSES = "0"  #@param {type:"string"}
#@markdown <small>Comma-separated class numbers, e.g. `0,2`.</small>
KMEANS_CLUSTERS = ""  #@param {type:"string"}
#@markdown <small>Optional finer selection within classes, using the k-means clusters of step 6 (see 7-b, colour by k-means): `1:3,7` keeps only clusters 3 and 7 of class 1; separate classes with `;`, e.g. `0:1,2; 1:5`. Classes not mentioned are kept whole.</small>
MIN_CLASS_PROBABILITY = 0.0  #@param {type:"slider", min:0, max:1, step:0.05}
NEW_DATASET_NAME = ""  #@param {type:"string"}
#@markdown <small>Blank = `<dataset>_classes_<list>`.</small>
MAKE_ACTIVE = True  #@param {type:"boolean"}
#@markdown <small>Use the subset as the default dataset for step 4.</small>
EXPORT_STAR = True  #@param {type:"boolean"}

if not SELECT_PARTICLES:
    print("Skipped (tick SELECT_PARTICLES to write a particle subset).")
else:
    import yaml
    require_install()
    exp_name, exp = get_experiment(EXPERIMENT)
    epoch = pick_epoch(exp, EPOCH)
    if epoch is None:
        raise RuntimeError("No checkpoint yet - train first (step 5).")
    out_dir = epoch_dir(exp, epoch)
    tcfg = yaml.safe_load((out_dir / "train-configs.yaml").read_text())
    if int(tcfg.get("n_classes", 1)) < 2:
        raise RuntimeError("This experiment has a single class - nothing to select.")
    p = load_pkl(out_dir / f"p_classes.{epoch}.pkl")
    pred = p.argmax(1)
    keep = [int(c) for c in KEEP_CLASSES.split(",") if c.strip()]
    mask = np.isin(pred, keep) & (p.max(1) >= MIN_CLASS_PROBABILITY)
    zdim = int(tcfg.get("z_dim", 0))
    if KMEANS_CLUSTERS.strip() and zdim < 2:
        raise RuntimeError("k-means clusters are only available for heterogeneous experiments with Z_DIM ≥ 2 "
                           "(Hydra does not save per-particle k-means labels otherwise); clear KMEANS_CLUSTERS.")
    for part in filter(None, (x.strip() for x in KMEANS_CLUSTERS.split(";"))):
        k, clusters = part.split(":")
        k, clusters = int(k), [int(c) for c in clusters.split(",") if c.strip()]
        if k not in keep:
            print(f"⚠️ KMEANS_CLUSTERS mentions class {k}, which is not in KEEP_CLASSES - ignoring it.")
            continue
        lab_file = newest((out_dir / f"analysis.epoch_{epoch}.class_{k}").glob("kmeans*/labels.pkl"))
        if lab_file is None:
            raise RuntimeError(f"No k-means labels for class {k} at epoch {epoch}: step 6 has not analysed this class "
                               "at this epoch (or skipped it for having too few particles).")
        in_k = np.nonzero(pred == k)[0]                   # k-means labels refer to the particles of class k only
        mask[in_k[~np.isin(load_pkl(lab_file), clusters)]] = False
    sel = np.nonzero(mask)[0]
    if tcfg.get("ind") is not None:                       # map back to the original particle file
        base = load_pkl(tcfg["ind"]) if isinstance(tcfg["ind"], str) else np.arange(int(tcfg["ind"]))
        sel = np.asarray(base)[sel]
    ds_name, ds = exp["dataset"], exp_dataset(exp)
    sel = sel.astype(np.int64)
    tag = "-".join(map(str, keep)) + ("_km" if KMEANS_CLUSTERS.strip() else "")
    new_name = NEW_DATASET_NAME or f"{ds_name}_classes_{tag}"
    sel_hash = hashlib.sha1(sel.tobytes()).hexdigest()[:6]     # different selections never overwrite each other
    ind_file = Path(exp["dir"]) / f"selected_classes_{tag}_epoch{epoch}_{sel_hash}.pkl"
    save_pkl(sel, ind_file)
    st = load_state()
    st["datasets"][new_name] = dict(ds, ind=str(ind_file), fingerprint=fingerprint(ds.get("fingerprint"), sel_hash))
    if MAKE_ACTIVE:
        st["active_dataset"] = new_name
    save_state(st)
    print(f"✅ Kept {len(sel)} of {len(p)} particles (classes {keep}"
          + (f", k-means {KMEANS_CLUSTERS}" if KMEANS_CLUSTERS.strip() else "") + f", p ≥ {MIN_CLASS_PROBABILITY}).")
    print(f"   Index file: {ind_file}")
    print(f"   Dataset '{new_name}'" + (" is now active - go back to step 4 to train on it." if MAKE_ACTIVE else " registered."))

    if EXPORT_STAR:
        src = ds.get("source_file")
        cu = q(str(BIN / "cryodrgn_utils"))
        if src and src.endswith(".star"):
            out = ind_file.with_suffix(".star")
            run(f"{cu} filter_star {q(src)} --ind {q(str(ind_file))} -o {q(str(out))}", quiet=True)
            print(f"   RELION particles: {out}  (same image paths as your original .star)")
        elif src and src.endswith(".cs"):
            out = ind_file.with_suffix(".cs")
            run(f"{cu} filter_cs {q(src)} --ind {q(str(ind_file))} -o {q(str(out))}", quiet=True)
            print(f"   cryoSPARC particles: {out}  (import it together with the original job's .csg, or convert it with pyem)")
        elif src:
            out = ind_file.with_suffix(".star")
            run(f"{cu} write_star {q(src)} --ctf {q(ds['ctf'])} --ind {q(str(ind_file))} -o {q(str(out))}", quiet=True)
            print(f"   RELION particles: {out}  (referencing your original stack {src})")
        else:
            print("   (No original particle file is recorded for this dataset, so no .star was written.)")

In [ ]:
#@title 8-b. Export: package results for download / Google Drive { display-mode: "form" }
EXPERIMENT = ""  #@param {type:"string"}
CONTENTS = "maps, plots and latents of one epoch"  #@param ["maps, plots and latents of one epoch", "everything except checkpoints", "everything"]
EPOCH = -1  #@param {type:"integer"}
#@markdown <small>For the first option: -1 = the analysed epoch (or the newest). Checkpoints (`weights.*.pkl`) are only needed to resume or re-analyse elsewhere.</small>
SET_VOXEL_SIZE = True  #@param {type:"boolean"}
#@markdown <small>Hydra writes maps with a voxel size of 1 Å; this rewrites every `.mrc` as a standard MRC file with the dataset's real pixel size (useful in ChimeraX).</small>
DOWNLOAD_ZIP = False  #@param {type:"boolean"}
COPY_TO_DRIVE = False  #@param {type:"boolean"}

exp_name, exp = get_experiment(EXPERIMENT)
exp_dir = Path(exp["dir"])
ds = exp_dataset(exp)
mrcs = sorted(exp_dir.glob("out*/**/*.mrc"))      # out/ and the out.<time> folders of resumed runs
if SET_VOXEL_SIZE:
    for f in mrcs:
        data, vs = read_mrc(f)
        if abs(vs - ds["apix"]) > 1e-3:
            write_mrc(f, data, ds["apix"])
    print(f"Voxel size {ds['apix']:.3f} Å set in {len(mrcs)} maps.")

staging = SCRATCH / "export_staging" / exp_name
shutil.rmtree(staging, ignore_errors=True)
if CONTENTS.startswith("maps"):
    epoch = pick_epoch(exp, EPOCH)
    if epoch is None:
        raise RuntimeError("No checkpoint yet - train first (step 5).")
    out_dir = epoch_dir(exp, epoch)
    (staging / "out").mkdir(parents=True)
    for f in [exp_dir / "configs.yaml", *(exp_dir.glob("selected_classes_*"))]:
        shutil.copy(f, staging / f.name)
    for pat in ("train-configs.yaml", "training.log", f"reconstruct.epoch_{epoch}.*", f"*.{epoch}.pkl"):
        for f in out_dir.glob(pat):
            if not f.name.startswith("weights."):
                shutil.copy(f, staging / "out" / f.name)
    for d in out_dir.glob(f"analysis.epoch_{epoch}.class_*"):
        shutil.copytree(d, staging / "out" / d.name)
    label = f"epoch{epoch}"
else:
    ignore = None if CONTENTS == "everything" else shutil.ignore_patterns("weights.*.pkl")
    shutil.copytree(exp_dir, staging, ignore=ignore)
    label = "all" if CONTENTS == "everything" else "no-checkpoints"
zip_base = WORK / "exports" / f"{exp_name}_{label}_{datetime.datetime.now():%Y%m%d-%H%M}"
archive = shutil.make_archive(str(zip_base), "zip", root_dir=staging.parent, base_dir=exp_name)
shutil.rmtree(staging, ignore_errors=True)
print(f"📦 {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")
if COPY_TO_DRIVE:
    if not IN_COLAB:
        print("Google Drive is only available in Colab.")
    elif on_drive(archive):
        print("The archive is already on Google Drive (WORK_DIR is on Drive).")
    else:
        if not os.path.ismount("/content/drive"):
            from google.colab import drive
            drive.mount("/content/drive")
        dest = Path("/content/drive/MyDrive/hydra_exports"); dest.mkdir(parents=True, exist_ok=True)
        shutil.copy(archive, dest)
        print(f"Copied to {dest / Path(archive).name}")
if DOWNLOAD_ZIP:
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    else:
        print("Browser download only works in Colab.")

In [ ]:
#@title (optional) Run any `drgnai` / `cryodrgn` command { display-mode: "form" }
#@markdown Runs inside the Hydra environment. `{EXP}` is replaced by the active experiment folder and `{WORK}` by `WORK_DIR`. Examples:
#@markdown * `drgnai analyze {EXP} --epoch 25 --class-idx 1 --trajectory-1d 0 50 10`: volumes interpolated between two particles of class 1 (indices count only the particles of that class)
#@markdown * `cryodrgn_utils filter_star /path/to/run_data.star --ind {EXP}/selected_classes_0_epoch25.pkl -o {WORK}/exports/class0.star` (8-a does this for you)
#@markdown * `cryodrgn --help`, `cryodrgn_utils --help`
COMMAND = "drgnai --help"  #@param {type:"string"}
require_install()
_exp_dir = get_experiment()[1]["dir"] if load_state().get("active_experiment") else ""
cmd = COMMAND.replace("{EXP}", q(_exp_dir)).replace("{WORK}", q(str(WORK)))
if cmd.startswith("drgnai "):        # use the CPU-safe wrapper from step 2
    cmd = q(str(DRGNAI)) + cmd[len("drgnai"):]
_ = run(cmd, check=False)

---
## 🩺 Troubleshooting & tips

| Symptom | What to do |
|:--|:--|
| **`CUDA out of memory`** | Pose search needs roughly 1.4 GB + 0.4 GB × `BATCH_SIZE_HPS` × `N_CLASSES`, and the SGD phase about 5 KB per pixel × `BATCH_SIZE_SGD` × `N_CLASSES` (128-wide network). Step 4 picks batch sizes that should fit; if not, set `BATCH_SIZE_HPS` / `BATCH_SIZE_SGD` lower by hand, reduce `N_CLASSES` or the box size, or use a GPU with more memory. |
| **RAM full / runtime crashes while loading data** | Hydra keeps all particles in memory (≈ 25–30 bytes × N × D² while loading); step 4 switches to lazy loading when that would not fit. Otherwise downsample, use a subset (`MAX_PARTICLES`), or tick `LAZY_LOADING` (slower, needs ≥ 1000 particles). |
| **Runtime disconnected during training** | With Google Drive, re-run steps 1–2, then step 5 (`START = auto` resumes). Without Drive, the outputs are lost. Keep a dashboard cell (5 or 5-b) running during long runs: idle runtimes are shut down. |
| **No GPU available any more (usage limits)** | You can still analyse and explore GPU-trained results on a CPU runtime (steps 1, 2, 6, 7). Training on a CPU needs `ALLOW_CPU_TRAINING` in step 5 and is very slow. |
| **Training is very slow** | Check that a GPU is attached (step 1). Pose search dominates, and its cost is proportional to particles × passes × `N_CLASSES`; the box size hardly matters for it. Use a random subset (`MAX_PARTICLES` in 3-B), fewer classes, a smaller `N_IMGS_POSE_SEARCH`, or a lower search resolution (`l_end: 24` in `EXTRA_CONFIG`). |
| **All particles end up in one class** | Train longer, try a higher `STD_NOISE` (more uniform class posteriors early on), or a different `SEED`. |
| **Maps look inverted / hollow in ChimeraX** | Hydra does not invert the particle contrast, so the sign of the maps follows your data (RELION/cryoSPARC "white" particles give positive maps). The viewers in step 7 fix the sign automatically. In ChimeraX, use `volume #1 level -0.01` or `volume scale #1 factor -1`. |
| **`drgnai` fails right after installation** | Re-run step 2 with `REINSTALL = True`. See `train_console.log` in the experiment folder for the full error. |
| **Want to use a different Hydra version** | Point `HYDRA_REPO` / `HYDRA_BRANCH` in step 2 to a fork or branch and tick `REINSTALL`. |

**Running other `drgnai` / `cryodrgn` commands:** use the cell above, or call the helper `run("drgnai ...")` from your own code cell; it always uses the Hydra environment.

**Viewing maps elsewhere:** download the export from step 8-b and open the `.mrc` files in [ChimeraX](https://www.cgl.ucsf.edu/chimerax/). Ab initio maps have arbitrary orientation and may have inverted handedness.

### References
* A. Levy\*, R. Raghu\*, D. Shustin\*, A. R.-Y. Peng, H. Li, O. B. Clarke, G. Wetzstein, E. D. Zhong. *Mixture of Neural Fields for Heterogeneous Reconstruction in Cryo-EM.* NeurIPS 2024.
* A. Levy, M. Grzadkowski *et al.* *Revealing biomolecular structure and motion with neural ab initio cryo-EM reconstruction* (DRGN-AI). bioRxiv 2024, doi:[10.1101/2024.05.30.596729](https://doi.org/10.1101/2024.05.30.596729).
* DRGN-AI documentation: <https://ez-lab.gitbook.io/drgn-ai/>